<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/03_deep_learning/architectures/cnn/experiment_cnn_mnist_progressive_improvement_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

"""
Insurance Claim Fraud Detection Using Graph Analysis

This script uses NetworkX to detect suspicious patterns in insurance claims data
by analyzing relationships between entities involved in claims.

Graph analysis is particularly useful in fraud detection because:
- Fraud rings often involve repeated connections between entities
"""

# Import required libraries for graph analysis
import pandas as pd
import networkx as nx
from pyvis.network import Network
from networkx.algorithms import community as nx_community
import numpy as np
import warnings
import webbrowser
import os

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')


def load_feature_engineered_dataset():
    """
    Load the feature-engineered insurance claims dataset.

    Returns:
        pd.DataFrame: Loaded dataset with engineered features
    """

    print("=" * 60)
    print("STEP 1: LOADING FEATURE ENGINEERED DATASET")
    print("=" * 60)

    # Define the path to the feature-engineered dataset
    dataset_path = "data/processed/insurance_claims_feature_engineered.csv"

    # Load the dataset from CSV file
    df = pd.read_csv(dataset_path)

    print(f"Dataset loaded successfully from: {dataset_path}")
    print(f"Dataset shape: {df.shape}")
    print(f"Number of claims: {len(df)}")
    print(f"Number of features: {df.shape[1]}")
    print()

    # Display basic statistics about fraud in the dataset
    fraud_count = df['fraud_reported'].sum()
    fraud_rate = (fraud_count / len(df)) * 100

    print(f"Fraud cases in dataset: {fraud_count}")
    print(f"Fraud rate: {fraud_rate:.1f}%")
    print()

    print("Graph analysis is useful in fraud detection because:")
    print("- Fraud rings often involve repeated connections between entities")
    print("- Collusive behavior creates unusual network patterns")
    print("- Traditional ML models may miss organized fraud networks")
    print("- Graph metrics can identify central players in fraud schemes")
    print()

    return df


def initialize_network_graph():
    """
    Initialize a NetworkX graph object for fraud detection.

    Returns:
        nx.Graph: Empty NetworkX graph object
    """

    print("=" * 60)
    print("STEP 2: INITIALIZING NETWORK GRAPH")
    print("=" * 60)

    # Create an undirected graph for fraud analysis
    # Undirected because relationships are bidirectional in fraud networks
    G = nx.Graph()

    print("NetworkX graph initialized successfully.")
    print("Graph type: Undirected")
    print("Nodes will represent entities such as:")
    print("  • Policy holders (policy_number)")
    print("  • Vehicles (auto_make + auto_model)")
    print("  • Incident locations (incident_location)")
    print("  • Geographic areas (insured_zip)")
    print()
    print("Edges will represent relationships between these entities.")
    print("Fraud rings may involve repeated connections between these entities.")
    print()

    return G


def create_nodes(graph, dataframe):
    """
    Create nodes for different entities in the insurance claims data with improved entity representation.

    Args:
        graph (nx.Graph): NetworkX graph object
        dataframe (pd.DataFrame): Insurance claims dataset

    Returns:
        dict: Dictionary mapping entity types to their node identifiers
    """

    print("=" * 60)
    print("STEP 3: CREATING NODES WITH IMPROVED ENTITY REPRESENTATION")
    print("=" * 60)

    # Initialize dictionary to store node identifiers by entity type
    entity_nodes = {
        'policy_holders': [],
        'vehicles': [],
        'locations': [],
        'zip_codes': []
    }

    # Create nodes for policy holders
    # Each policy_number represents a unique policy holder
    for policy_num in dataframe['policy_number'].unique():
        # Create node with policy number as identifier
        node_id = f"policy_{policy_num}"
        graph.add_node(node_id,
                      entity_type='policy_holder',
                      policy_number=policy_num,
                      shared_entity_score=0)  # Initialize shared entity score
        entity_nodes['policy_holders'].append(node_id)

    print(f"Created {len(entity_nodes['policy_holders'])} policy holder nodes")

    # Create nodes for vehicles with improved identifiers
    # Use auto_make + auto_model + auto_year for unique vehicle identification
    for _, row in dataframe.iterrows():
        # Create unique vehicle identifier with year for better distinction
        vehicle_id = f"vehicle_{row['auto_make']}_{row['auto_model']}_{row['auto_year']}"

        # Add node if it doesn't exist already
        if not graph.has_node(vehicle_id):
            graph.add_node(vehicle_id,
                          entity_type='vehicle',
                          vehicle_make=row['auto_make'],
                          vehicle_model=row['auto_model'],
                          vehicle_year=row['auto_year'],
                          shared_entity_score=0)  # Initialize shared entity score
            entity_nodes['vehicles'].append(vehicle_id)

    print(f"Created {len(entity_nodes['vehicles'])} unique vehicle nodes")

    # Create nodes for incident locations
    # Each unique incident_location represents a specific accident location
    for location in dataframe['incident_location'].unique():
        if pd.notna(location):  # Skip null values
            # Create node with location as identifier
            location_id = f"location_{location}"
            graph.add_node(location_id,
                          entity_type='location',
                          incident_location=location,
                          shared_entity_score=0)  # Initialize shared entity score
            entity_nodes['locations'].append(location_id)

    print(f"Created {len(entity_nodes['locations'])} incident location nodes")

    # Create nodes for ZIP codes
    # Each insured_zip represents a geographic area
    for zip_code in dataframe['insured_zip'].unique():
        if pd.notna(zip_code):  # Skip null values
            # Create node with ZIP code as identifier
            zip_id = f"zip_{zip_code}"
            graph.add_node(zip_id,
                          entity_type='zip_code',
                          insured_zip=zip_code,
                          shared_entity_score=0)  # Initialize shared entity score
            entity_nodes['zip_codes'].append(zip_id)

    print(f"Created {len(entity_nodes['zip_codes'])} ZIP code nodes")
    print()

    total_nodes = graph.number_of_nodes()
    print(f"Total nodes created: {total_nodes}")
    print()
    print("Improved entity representation:")
    print("- Vehicles: make_model_year (e.g., Toyota_Camry_2018)")
    print("- Policy holders: policy_number")
    print("- Locations: incident_location")
    print("- ZIP codes: insured_zip")
    print()
    print("Each node now stores comprehensive attributes for fraud investigation.")
    print()

    return entity_nodes


def add_edges_based_on_relationships(graph, dataframe):
    """
    Add edges to represent relationships between entities in claims.
    Also compute shared entity scores to identify potential fraud rings.

    Args:
        graph (nx.Graph): NetworkX graph object
        dataframe (pd.DataFrame): Insurance claims dataset
    """

    print("=" * 60)
    print("STEP 4: ADDING EDGES BASED ON RELATIONSHIPS")
    print("=" * 60)

    edge_count = 0

    # Track entity usage for shared entity score computation
    vehicle_usage = {}
    location_usage = {}
    zip_usage = {}

    # Add edges between policy holders and vehicles
    # Each claim connects a policy holder to a specific vehicle
    for _, row in dataframe.iterrows():
        # Create identifiers for policy holder and vehicle with new format
        policy_id = f"policy_{row['policy_number']}"
        vehicle_id = f"vehicle_{row['auto_make']}_{row['auto_model']}_{row['auto_year']}"

        # Track vehicle usage for shared entity score
        if vehicle_id not in vehicle_usage:
            vehicle_usage[vehicle_id] = []
        vehicle_usage[vehicle_id].append(policy_id)

        # Add edge if both nodes exist
        if graph.has_node(policy_id) and graph.has_node(vehicle_id):
            # Add edge with claim amount as weight
            graph.add_edge(policy_id, vehicle_id,
                          relationship='policy_vehicle',
                          claim_amount=row['total_claim_amount'],
                          fraud_flag=row['fraud_reported'])
            edge_count += 1

    print(f"Added {edge_count} policy-to-vehicle edges")

    # Add edges between policy holders and incident locations
    # Each claim connects a policy holder to a specific location
    location_edge_count = 0
    for _, row in dataframe.iterrows():
        # Create identifiers for policy holder and location
        policy_id = f"policy_{row['policy_number']}"

        # Skip if incident location is null
        if pd.isna(row['incident_location']):
            continue

        location_id = f"location_{row['incident_location']}"

        # Track location usage for shared entity score
        if location_id not in location_usage:
            location_usage[location_id] = []
        location_usage[location_id].append(policy_id)

        # Add edge if both nodes exist
        if graph.has_node(policy_id) and graph.has_node(location_id):
            # Add edge with claim amount as weight
            graph.add_edge(policy_id, location_id,
                          relationship='policy_location',
                          claim_amount=row['total_claim_amount'],
                          fraud_flag=row['fraud_reported'])
            location_edge_count += 1

    print(f"Added {location_edge_count} policy-to-location edges")

    # Add edges between policy holders and ZIP codes
    # Each policy holder is associated with a specific ZIP code
    zip_edge_count = 0
    for _, row in dataframe.iterrows():
        # Create identifiers for policy holder and ZIP code
        policy_id = f"policy_{row['policy_number']}"

        # Skip if ZIP code is null
        if pd.isna(row['insured_zip']):
            continue

        zip_id = f"zip_{row['insured_zip']}"

        # Track ZIP usage for shared entity score
        if zip_id not in zip_usage:
            zip_usage[zip_id] = []
        zip_usage[zip_id].append(policy_id)

        # Add edge if both nodes exist
        if graph.has_node(policy_id) and graph.has_node(zip_id):
            # Add edge with claim amount as weight
            graph.add_edge(policy_id, zip_id,
                          relationship='policy_zip',
                          claim_amount=row['total_claim_amount'],
                          fraud_flag=row['fraud_reported'])
            zip_edge_count += 1

    print(f"Added {zip_edge_count} policy-to-ZIP edges")
    print()

    # Compute shared entity scores for fraud ring detection
    print("COMPUTING SHARED ENTITY SCORES:")
    print("-" * 40)

    # Update shared entity scores for vehicles
    for vehicle_id, policy_holders in vehicle_usage.items():
        if len(policy_holders) > 1:  # Vehicle used by multiple policy holders
            shared_score = len(policy_holders) - 1  # Score based on number of additional users
            if graph.has_node(vehicle_id):
                graph.nodes[vehicle_id]['shared_entity_score'] = shared_score

    high_score_vehicles = sum(1 for v in vehicle_usage.values() if len(v) > 1)
    print(f"Vehicles shared by multiple policy holders: {high_score_vehicles}")

    # Update shared entity scores for locations
    for location_id, policy_holders in location_usage.items():
        if len(policy_holders) > 1:  # Location used by multiple policy holders
            shared_score = len(policy_holders) - 1
            if graph.has_node(location_id):
                graph.nodes[location_id]['shared_entity_score'] = shared_score

    high_score_locations = sum(1 for l in location_usage.values() if len(l) > 1)
    print(f"Locations shared by multiple policy holders: {high_score_locations}")

    # Update shared entity scores for ZIP codes
    for zip_id, policy_holders in zip_usage.items():
        if len(policy_holders) > 1:  # ZIP connects multiple policy holders
            shared_score = len(policy_holders) - 1
            if graph.has_node(zip_id):
                graph.nodes[zip_id]['shared_entity_score'] = shared_score

    high_score_zips = sum(1 for z in zip_usage.values() if len(z) > 1)
    print(f"ZIP codes with multiple policy holders: {high_score_zips}")
    print()

    total_edges = graph.number_of_edges()
    print(f"Total edges in graph: {total_edges}")
    print()
    print("Shared entity scores help identify fraud rings where:")
    print("- Multiple policy holders use the same vehicle")
    print("- Multiple incidents occur at the same location")
    print("- Many policy holders share the same geographic area")
    print()


def compute_graph_metrics(graph):
    """
    Compute centrality metrics to identify important nodes in the fraud network.

    Args:
        graph (nx.Graph): NetworkX graph object

    Returns:
        dict: Dictionary containing centrality metrics for all nodes
    """

    print("=" * 60)
    print("STEP 5: COMPUTING GRAPH METRICS")
    print("=" * 60)

    # Compute degree centrality
    # Measures how many connections each node has
    degree_centrality = nx.degree_centrality(graph)

    # Compute betweenness centrality
    # Measures how often a node appears on shortest paths between other nodes
    betweenness_centrality = nx.betweenness_centrality(graph)

    # Compute eigenvector centrality
    # Measures influence of a node based on its connections to other influential nodes
    try:
        eigenvector_centrality = nx.eigenvector_centrality(graph)
    except:
        # Handle cases where eigenvector centrality fails to converge
        eigenvector_centrality = {node: 0 for node in graph.nodes()}

    print("Graph metrics computed successfully:")
    print(f"  • Degree centrality: Connection frequency")
    print(f"  • Betweenness centrality: Bridge/broker importance")
    print(f"  • Eigenvector centrality: Influence in network")
    print()

    # Store all metrics in a dictionary
    centrality_metrics = {
        'degree': degree_centrality,
        'betweenness': betweenness_centrality,
        'eigenvector': eigenvector_centrality
    }

    print("Why these metrics help identify suspicious entities:")
    print("High centrality nodes may indicate:")
    print("  • Frequently involved vehicles")
    print("  • Repeated incident locations")
    print("  • Policy holders linked to many claims")
    print("  • Central players in fraud networks")
    print()

    return centrality_metrics


def detect_suspicious_nodes(graph, centrality_metrics):
    """
    Identify nodes with unusually high centrality values.
    Store top suspicious nodes for investigation subgraph creation.

    Args:
        graph (nx.Graph): NetworkX graph object
        centrality_metrics (dict): Dictionary containing centrality metrics

    Returns:
        dict: Dictionary containing suspicious nodes by metric and entity type
        list: Top suspicious nodes for investigation
    """

    print("=" * 60)
    print("STEP 6: DETECTING SUSPICIOUS NODES")
    print("=" * 60)

    # Define percentile threshold for identifying suspicious nodes
    percentile_threshold = 95  # Top 5% most central nodes

    suspicious_nodes = {
        'degree': {},
        'betweenness': {},
        'eigenvector': {}
    }

    # Collect all suspicious nodes for investigation subgraph
    top_suspicious_nodes = []

    # Analyze degree centrality
    degree_values = list(centrality_metrics['degree'].values())
    degree_threshold = np.percentile(degree_values, percentile_threshold)

    print(f"Degree centrality threshold (95th percentile): {degree_threshold:.3f}")

    # Find nodes with high degree centrality
    for node, centrality_value in centrality_metrics['degree'].items():
        if centrality_value >= degree_threshold:
            # Get node attributes to determine entity type
            node_attrs = graph.nodes[node]
            entity_type = node_attrs.get('entity_type', 'unknown')

            if entity_type not in suspicious_nodes['degree']:
                suspicious_nodes['degree'][entity_type] = []

            node_info = {
                'node': node,
                'centrality': centrality_value,
                'attributes': node_attrs
            }
            suspicious_nodes['degree'][entity_type].append(node_info)
            top_suspicious_nodes.append(node)

    # Analyze betweenness centrality
    betweenness_values = list(centrality_metrics['betweenness'].values())
    betweenness_threshold = np.percentile(betweenness_values, percentile_threshold)

    print(f"Betweenness centrality threshold (95th percentile): {betweenness_threshold:.3f}")

    # Find nodes with high betweenness centrality
    for node, centrality_value in centrality_metrics['betweenness'].items():
        if centrality_value >= betweenness_threshold:
            # Get node attributes to determine entity type
            node_attrs = graph.nodes[node]
            entity_type = node_attrs.get('entity_type', 'unknown')

            if entity_type not in suspicious_nodes['betweenness']:
                suspicious_nodes['betweenness'][entity_type] = []

            node_info = {
                'node': node,
                'centrality': centrality_value,
                'attributes': node_attrs
            }
            suspicious_nodes['betweenness'][entity_type].append(node_info)
            top_suspicious_nodes.append(node)

    # Analyze eigenvector centrality
    eigenvector_values = list(centrality_metrics['eigenvector'].values())
    eigenvector_threshold = np.percentile(eigenvector_values, percentile_threshold)

    print(f"Eigenvector centrality threshold (95th percentile): {eigenvector_threshold:.3f}")

    # Find nodes with high eigenvector centrality
    for node, centrality_value in centrality_metrics['eigenvector'].items():
        if centrality_value >= eigenvector_threshold:
            # Get node attributes to determine entity type
            node_attrs = graph.nodes[node]
            entity_type = node_attrs.get('entity_type', 'unknown')

            if entity_type not in suspicious_nodes['eigenvector']:
                suspicious_nodes['eigenvector'][entity_type] = []

            node_info = {
                'node': node,
                'centrality': centrality_value,
                'attributes': node_attrs
            }
            suspicious_nodes['eigenvector'][entity_type].append(node_info)
            top_suspicious_nodes.append(node)

    # Remove duplicates and get unique top suspicious nodes
    top_suspicious_nodes = list(set(top_suspicious_nodes))

    print()
    print(f"Suspicious nodes identified based on high connectivity: {len(top_suspicious_nodes)} total")
    print("Unusually high connectivity may indicate organized fraud rings.")
    print()

    return suspicious_nodes, top_suspicious_nodes


def detect_suspicious_clusters(graph):
    """
    Use community detection to identify tightly connected clusters.

    Args:
        graph (nx.Graph): NetworkX graph object

    Returns:
        dict: Dictionary containing cluster information
    """

    print("=" * 60)
    print("STEP 7: DETECTING SUSPICIOUS CLUSTERS")
    print("=" * 60)

    # Use the Louvain algorithm for community detection
    # This algorithm finds densely connected groups of nodes
    try:
        detected_communities = nx_community.louvain_communities(graph, seed=42)
    except:
        # Fallback to greedy modularity communities if Louvain fails
        detected_communities = nx_community.greedy_modularity_communities(graph)

    print(f"Number of communities detected: {len(detected_communities)}")

    # Analyze community sizes
    community_sizes = [len(community) for community in detected_communities]
    print(f"Community size range: {min(community_sizes)} to {max(community_sizes)}")
    print(f"Average community size: {np.mean(community_sizes):.1f}")

    # Identify suspicious clusters (unusually large communities)
    size_threshold = np.percentile(community_sizes, 90)  # Top 10% largest communities
    suspicious_clusters = []

    for i, community in enumerate(detected_communities):
        if len(community) >= size_threshold:
            # Analyze entity types in this community
            entity_types = {}
            for node in community:
                node_attrs = graph.nodes[node]
                entity_type = node_attrs.get('entity_type', 'unknown')
                entity_types[entity_type] = entity_types.get(entity_type, 0) + 1

            suspicious_clusters.append({
                'cluster_id': i,
                'size': len(community),
                'nodes': list(community),
                'entity_types': entity_types
            })

    print(f"Number of suspicious clusters (top 10% largest): {len(suspicious_clusters)}")
    print()
    print("How fraud rings may appear as tightly connected clusters:")
    print("- Groups of policy holders sharing vehicles")
    print("- Clusters of claims at same locations")
    print("- Networks of entities in geographic proximity")
    print()

    cluster_info = {
        'all_communities': list(detected_communities),  # Convert to list
        'suspicious_clusters': suspicious_clusters,
        'size_threshold': size_threshold
    }

    return cluster_info

def print_suspicious_entities(suspicious_nodes, suspicious_clusters):
    """
    Display the most suspicious entities identified in the analysis.

    Args:
        suspicious_nodes (dict): Dictionary containing suspicious nodes
        suspicious_clusters (dict): Dictionary containing suspicious clusters
    """

    print("=" * 60)
    print("STEP 8: PRINTING SUSPICIOUS ENTITIES")
    print("=" * 60)

    # Print top suspicious policy holders
    print("TOP SUSPICIOUS POLICY HOLDERS:")
    print("-" * 40)

    if 'policy_holder' in suspicious_nodes['degree']:
        # Sort by degree centrality
        policy_holders = sorted(suspicious_nodes['degree']['policy_holder'],
                              key=lambda x: x['centrality'], reverse=True)

        for i, holder in enumerate(policy_holders[:5], 1):  # Top 5
            policy_num = holder['attributes']['policy_number']
            centrality = holder['centrality']
            print(f"  {i}. Policy {policy_num} - Degree Centrality: {centrality:.3f}")
    else:
        print("  No suspicious policy holders detected by degree centrality.")

    print()

    # Print top suspicious vehicles
    print("TOP SUSPICIOUS VEHICLES:")
    print("-" * 40)

    if 'vehicle' in suspicious_nodes['degree']:
        # Sort by degree centrality
        vehicles = sorted(suspicious_nodes['degree']['vehicle'],
                        key=lambda x: x['centrality'], reverse=True)

        for i, vehicle in enumerate(vehicles[:5], 1):  # Top 5
            make = vehicle['attributes']['vehicle_make']
            model = vehicle['attributes']['vehicle_model']
            year = vehicle['attributes']['vehicle_year']
            centrality = vehicle['centrality']
            print(f"  {i}. {make} {model} {year} - Degree Centrality: {centrality:.3f}")
    else:
        print("  No suspicious vehicles detected by degree centrality.")

    print()

    # Print top suspicious locations
    print("TOP SUSPICIOUS LOCATIONS:")
    print("-" * 40)

    if 'location' in suspicious_nodes['degree']:
        # Sort by degree centrality
        locations = sorted(suspicious_nodes['degree']['location'],
                         key=lambda x: x['centrality'], reverse=True)

        for i, location in enumerate(locations[:5], 1):  # Top 5
            loc = location['attributes']['incident_location']
            centrality = location['centrality']
            print(f"  {i}. Location {loc} - Degree Centrality: {centrality:.3f}")
    else:
        print("  No suspicious locations detected by degree centrality.")

    print()

    # Print suspicious clusters
    print("SUSPICIOUS CLUSTERS:")
    print("-" * 40)

    if suspicious_clusters['suspicious_clusters']:
        # Sort by cluster size
        clusters = sorted(suspicious_clusters['suspicious_clusters'],
                         key=lambda x: x['size'], reverse=True)

        for i, cluster in enumerate(clusters[:5], 1):  # Top 5
            print(f"  {i}. Cluster {cluster['cluster_id']}:")
            print(f"     Size: {cluster['size']} nodes")
            print(f"     Entity types: {cluster['entity_types']}")
    else:
        print("  No suspicious clusters detected.")

    print()


def create_interactive_investigation_graph(graph, top_suspicious_nodes, centrality_metrics, max_nodes=100):
    """
    Create an investigator-friendly interactive fraud investigation graph.

    Args:
        graph (nx.Graph): NetworkX graph object
        top_suspicious_nodes (list): List of top suspicious nodes
        centrality_metrics (dict): Centrality metrics for all nodes
        max_nodes (int): Maximum number of nodes for the investigation graph
    """

    print("=" * 60)
    print("STEP 9: CREATING INVESTIGATOR-FRIENDLY FRAUD INVESTIGATION GRAPH")
    print("=" * 60)

    # Build investigation subgraph with top 20 suspicious nodes and their neighbors
    nodes_to_include = set()

    # Take top 20 most suspicious nodes (prioritize by degree centrality)
    suspicious_with_centrality = []
    for node in top_suspicious_nodes:
        if node in centrality_metrics['degree']:
            suspicious_with_centrality.append((node, centrality_metrics['degree'][node]))

    # Sort by centrality and take top 20
    suspicious_with_centrality.sort(key=lambda x: x[1], reverse=True)
    top_20_suspicious = [node for node, _ in suspicious_with_centrality[:20]]
    nodes_to_include.update(top_20_suspicious)

    # Add immediate neighbors of top suspicious nodes
    for suspicious_node in top_20_suspicious:
        if suspicious_node in graph.nodes():
            neighbors = list(graph.neighbors(suspicious_node))
            # Add up to 5 neighbors per suspicious node
            nodes_to_include.update(neighbors[:5])

    # Limit total nodes to prevent clutter
    if len(nodes_to_include) > max_nodes:
        nodes_to_include = set(list(nodes_to_include)[:max_nodes])

    # Create investigation subgraph
    investigation_subgraph = graph.subgraph(nodes_to_include)

    print(f"Building investigation graph with {len(nodes_to_include)} nodes")
    print(f"Including top 20 suspicious nodes and their immediate connections")
    print()

    # Initialize PyVis network with investigator-friendly configuration
    pyvis_network = Network(
        height='900px',
        width='1400px',
        bgcolor='#f8f9fa',
        font_color='black',
        notebook=False,
        cdn_resources='in_line',
        select_menu=True,
        filter_menu=True
    )

    # Configure physics for better layout using forceAtlas2Based
    pyvis_network.set_options("""
    var options = {
      "physics": {
        "enabled": true,
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.01,
          "springLength": 100,
          "springConstant": 0.08,
          "damping": 0.4,
          "avoidOverlap": 0.5
        },
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based"
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 200,
        "zoomSpeed": 0.5,
        "dragNodes": true,
        "dragView": true,
        "zoomView": true
      },
      "layout": {
        "improvedLayout": true
      }
    }
    """)

    # Identify suspicious nodes for highlighting
    suspicious_set = set(top_suspicious_nodes)

    # Add nodes with realistic entity appearance and hover information
    for node in investigation_subgraph.nodes():
        node_attrs = graph.nodes[node]
        entity_type = node_attrs.get('entity_type', 'unknown')

        # Determine node appearance and icon based on entity type
        if entity_type == 'policy_holder':
            color = '#3498db'  # Blue
            shape = 'icon'
            icon = {'face': 'FontAwesome', 'code': 'uf007'}  # User/person icon
            policy_num = node_attrs.get('policy_number', 'Unknown')
            label = f"Policy {policy_num}"

            # Create investigator-friendly hover information
            degree_centrality = centrality_metrics['degree'].get(node, 0)
            shared_score = node_attrs.get('shared_entity_score', 0)
            connections = investigation_subgraph.degree(node)

            # Determine risk level based on centrality
            if degree_centrality > 0.002:
                risk_level = "HIGH"
            elif degree_centrality > 0.001:
                risk_level = "MEDIUM"
            else:
                risk_level = "LOW"

            # Determine network importance
            if connections > 6:
                importance = "Very High"
            elif connections > 4:
                importance = "High"
            elif connections > 2:
                importance = "Medium"
            else:
                importance = "Low"

            hover_info = f"""POLICY HOLDER INVESTIGATION
─────────────────────────
Policy Number: {policy_num}
Connected Entities: {connections}
Network Influence: {importance}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level}

{'⚠️ HIGH RISK - Investigate Priority' if risk_level == 'HIGH' else '✅ Standard Risk Level'}

Investigation Notes:
• Check for unusual claim patterns
• Verify vehicle ownership legitimacy
• Cross-reference with incident locations"""

        elif entity_type == 'vehicle':
            color = '#27ae60'  # Green
            shape = 'icon'
            icon = {'face': 'FontAwesome', 'code': 'uf1b9'}  # Car icon

            make = node_attrs.get('vehicle_make', 'Unknown')
            model = node_attrs.get('vehicle_model', 'Unknown')
            year = node_attrs.get('vehicle_year', 'Unknown')
            label = f"Vehicle {make} {model}"

            # Create investigator-friendly hover information
            degree_centrality = centrality_metrics['degree'].get(node, 0)
            shared_score = node_attrs.get('shared_entity_score', 0)
            connections = investigation_subgraph.degree(node)

            # Determine risk level for vehicles
            if connections > 3:
                risk_level = "HIGH"
            elif connections > 2:
                risk_level = "MEDIUM"
            else:
                risk_level = "LOW"

            hover_info = f"""VEHICLE INVESTIGATION
─────────────────────────
Vehicle: {make} {model} ({year})
Number of Claims: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level}

{'⚠️ SUSPICIOUS - Multiple Policy Holders' if connections > 3 else '✅ Normal Usage Pattern'}

Investigation Notes:
• Vehicle used by {connections} different policy holders
• High shared usage may indicate fraud ring
• Verify vehicle identification numbers
• Check for staged accidents"""

        elif entity_type == 'location':
            color = '#e74c3c'  # Red
            shape = 'icon'
            icon = {'face': 'FontAwesome', 'code': 'uf3c5'}  # Map marker icon

            location = node_attrs.get('incident_location', 'Unknown')
            label = f"Location {location}"

            # Create investigator-friendly hover information
            degree_centrality = centrality_metrics['degree'].get(node, 0)
            shared_score = node_attrs.get('shared_entity_score', 0)
            connections = investigation_subgraph.degree(node)

            # Determine risk level for locations
            if connections > 3:
                risk_level = "HIGH"
            elif connections > 2:
                risk_level = "MEDIUM"
            else:
                risk_level = "LOW"

            hover_info = f"""LOCATION INVESTIGATION
─────────────────────────
Location: {location}
Number of Incidents: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level}

{'⚠️ SUSPICIOUS - Repeated Accident Location' if connections > 3 else '✅ Normal Incident Frequency'}

Investigation Notes:
• Multiple accidents at same location
• May indicate staged accident location
• Verify location legitimacy
• Check for repair shop connections"""

        elif entity_type == 'zip_code':
            color = '#f39c12'  # Orange
            shape = 'icon'
            icon = {'face': 'FontAwesome', 'code': 'uf124'}  # Location pin icon

            zip_code = node_attrs.get('insured_zip', 'Unknown')
            label = f"ZIP {zip_code}"

            # Create investigator-friendly hover information
            degree_centrality = centrality_metrics['degree'].get(node, 0)
            shared_score = node_attrs.get('shared_entity_score', 0)
            connections = investigation_subgraph.degree(node)

            # Determine risk level for ZIP codes
            if connections > 10:
                risk_level = "HIGH"
            elif connections > 6:
                risk_level = "MEDIUM"
            else:
                risk_level = "LOW"

            hover_info = f"""GEOGRAPHIC AREA INVESTIGATION
─────────────────────────
ZIP Code: {zip_code}
Policy Holders: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level}

{'⚠️ SUSPICIOUS - High Density Area' if connections > 10 else '✅ Normal Population Density'}

Investigation Notes:
• High concentration of policy holders
• May indicate geographic fraud ring
• Cross-reference with incident locations
• Check for organized fraud patterns"""
        else:
            color = '#95a5a6'  # Gray for unknown
            shape = 'dot'
            label = str(node)
            icon = None
            hover_info = f"Unknown Entity: {node}"

        # Determine node size based on suspiciousness and centrality
        base_size = 20
        if node in suspicious_set:
            # Suspicious nodes are larger
            centrality_score = centrality_metrics['degree'].get(node, 0)
            node_size = base_size + (centrality_score * 100) + 30
            border_width = 3
            border_color = '#e74c3c'  # Red border for suspicious
        else:
            # Normal nodes are smaller
            centrality_score = centrality_metrics['degree'].get(node, 0)
            node_size = base_size + (centrality_score * 50)
            border_width = 1
            border_color = '#34495e'  # Dark gray border

        # Add node to PyVis network with all attributes in single call
        pyvis_network.add_node(
            node,
            label=label,
            color=color,
            shape=shape,
            size=node_size,
            title=hover_info,  # Hover information
            borderWidth=border_width,
            borderColor=border_color,
            font={'size': 12, 'color': 'black'},
            chosen=True,  # Allow node selection
            icon=icon  # Include icon for vehicles and locations
        )

    # Add edges with relationship information
    for edge in investigation_subgraph.edges(data=True):
        source, target, edge_data = edge
        relationship = edge_data.get('relationship', 'unknown')
        claim_amount = edge_data.get('claim_amount', 0)
        fraud_flag = edge_data.get('fraud_flag', 0)

        # Create investigator-friendly edge descriptions
        if relationship == 'owns_vehicle':
            edge_description = "Vehicle used in claim"
        elif relationship == 'incident_at':
            edge_description = "Accident location"
        elif relationship == 'resides_in':
            edge_description = "Residence area"
        else:
            edge_description = relationship.replace('_', ' ').title()

        # Create edge hover information
        edge_hover = f"""RELATIONSHIP DETAILS
─────────────────────────
Type: {edge_description}
Claim Amount: ${claim_amount:,.2f}
Fraud Flag: {'⚠️ FRAUD DETECTED' if fraud_flag else '✅ No Fraud Flag'}

Investigation Notes:
• Verify legitimacy of this connection
• Check for unusual patterns
• Cross-reference with other claims"""

        # Style edges based on fraud flag
        if fraud_flag:
            edge_color = '#e74c3c'  # Red for fraud
            edge_width = 3
        else:
            edge_color = '#95a5a6'  # Gray for normal
            edge_width = 1

        pyvis_network.add_edge(
            source,
            target,
            color=edge_color,
            width=edge_width,
            title=edge_hover,
            hoverWidth=3
        )

    # Add interactive features and save the investigation graph
    print("Interactive investigation features:")
    print("- Zoom and pan capabilities")
    print("- Drag nodes to rearrange layout")
    print("- Hover over nodes for detailed information")
    print("- Click and drag to select multiple nodes")
    print("- Use filter menu to show/hide entity types")
    print()

    # Generate investigation insights for dashboard
    total_policies = len([n for n in investigation_subgraph.nodes() if graph.nodes[n].get('entity_type') == 'policy_holder'])
    suspicious_policies = len([n for n in investigation_subgraph.nodes() if n in suspicious_set and graph.nodes[n].get('entity_type') == 'policy_holder'])
    repeated_vehicles = len([n for n in investigation_subgraph.nodes() if graph.nodes[n].get('entity_type') == 'vehicle' and investigation_subgraph.degree(n) > 2])
    repeated_locations = len([n for n in investigation_subgraph.nodes() if graph.nodes[n].get('entity_type') == 'location' and investigation_subgraph.degree(n) > 2])

    # Save the interactive HTML file
    output_file = "fraud_network_investigation.html"
    try:
        # Generate HTML and add investigation dashboard
        html_content = pyvis_network.generate_html()

        # Create investigation dashboard
        investigation_dashboard = f"""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 15px; margin: 10px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.3);">
        <h1 style="text-align: center; margin: 0 0 20px 0; font-size: 2.5em; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
            🔍 FRAUD INVESTIGATION DASHBOARD
        </h1>

        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
            <!-- Investigation Insights Panel -->
            <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
                <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">📊 INVESTIGATION INSIGHTS</h3>
                <div style="font-size: 0.95em; line-height: 1.6;">
                    <div style="margin-bottom: 8px;"><strong>Total Policies Analyzed:</strong> {total_policies}</div>
                    <div style="margin-bottom: 8px;"><strong>Suspicious Policies:</strong> <span style="color: #ff6b6b; font-weight: bold;">{suspicious_policies}</span></div>
                    <div style="margin-bottom: 8px;"><strong>Repeated Vehicles Detected:</strong> <span style="color: #ffd93d; font-weight: bold;">{repeated_vehicles}</span></div>
                    <div style="margin-bottom: 8px;"><strong>Repeated Locations Detected:</strong> <span style="color: #6bcf7f; font-weight: bold;">{repeated_locations}</span></div>
                </div>
            </div>

            <!-- Graph Legend Panel -->
            <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
                <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">📖 GRAPH LEGEND</h3>
                <div style="font-size: 0.9em; line-height: 1.5;">
                    <div style="margin-bottom: 6px;"><span style="color: #3498db; font-weight: bold;">👤 Blue nodes</span> → Policy Holders</div>
                    <div style="margin-bottom: 6px;"><span style="color: #27ae60; font-weight: bold;">🚗 Green nodes</span> → Vehicles</div>
                    <div style="margin-bottom: 6px;"><span style="color: #e74c3c; font-weight: bold;">📍 Red nodes</span> → Accident Locations</div>
                    <div style="margin-bottom: 6px;"><span style="color: #f39c12; font-weight: bold;">📮 Orange nodes</span> → ZIP Codes</div>
                    <div style="margin-bottom: 6px;"><span style="color: #e74c3c; font-weight: bold;">🔴 Red edges</span> → Fraud flagged claims</div>
                    <div style="margin-bottom: 6px;"><span style="color: #95a5a6; font-weight: bold;">⚪ Gray edges</span> → Normal claims</div>
                    <div style="margin-top: 8px; padding: 6px; background: rgba(255,255,255,0.2); border-radius: 5px;">
                        <strong>Large nodes</strong> = Frequently appearing<br>
                        <strong>Red borders</strong> = High fraud risk
                    </div>
                </div>
            </div>
        </div>

        <!-- Investigation Explanation Panel -->
        <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
            <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">🎯 HOW TO INVESTIGATE</h3>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; font-size: 0.9em; line-height: 1.5;">
                <div>
                    <h4 style="color: #6bcf7f; margin: 0 0 8px 0;">Understanding the Graph:</h4>
                    <ul style="margin: 0; padding-left: 20px;">
                        <li>Clusters represent connected claims</li>
                        <li>Shared entities may indicate fraud rings</li>
                        <li>Node size shows frequency of appearance</li>
                        <li>Red borders highlight high-risk entities</li>
                    </ul>
                </div>
                <div>
                    <h4 style="color: #6bcf7f; margin: 0 0 8px 0;">Investigation Actions:</h4>
                    <ul style="margin: 0; padding-left: 20px;">
                        <li>🖱️ <strong>Hover</strong> for detailed investigation info</li>
                        <li>🔍 <strong>Zoom</strong> to examine specific areas</li>
                        <li>✋ <strong>Drag</strong> nodes to rearrange layout</li>
                        <li>🎯 <strong>Click</strong> to highlight connections</li>
                    </ul>
                </div>
            </div>
        </div>
    </div>

    <!-- FontAwesome CDN for proper icon rendering -->
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
        """

        # Insert FontAwesome CDN in head
        head_index = html_content.find('<head>')
        if head_index != -1:
            head_end = html_content.find('</head>', head_index) + 8
            font_awesome_cdn = '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">'
            html_content = html_content[:head_end] + font_awesome_cdn + html_content[head_end:]

        # Insert investigation dashboard after the body tag
        body_index = html_content.find('<body')
        if body_index != -1:
            # Find the end of the body opening tag
            body_end = html_content.find('>', body_index) + 1
            # Insert the dashboard
            html_content = html_content[:body_end] + investigation_dashboard + html_content[body_end:]

        # Save with UTF-8 encoding
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html_content)
        print(f"✅ Investigator-friendly fraud dashboard saved as: {output_file}")
    except Exception as e:
        # Fallback: try saving with ASCII-safe name
        output_file = "fraud_network.html"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(pyvis_network.generate_html())
        print(f"⚠️ Investigation dashboard saved as: {output_file}")

    # Auto-open the graph in default browser
    try:
        file_path = os.path.abspath(output_file)
        webbrowser.open(f'file://{file_path}')
        print(f"🌐 Investigation dashboard automatically opened in your browser")
    except Exception as e:
        print(f"⚠️ Could not auto-open browser. Please open {output_file} manually.")
    print()
    print("🔍 INVESTIGATION CHECKLIST:")
    print("─────────────────────────")
    print("1. 🚗 Look for vehicles connected to multiple policy holders")
    print("2. 📍 Check locations with repeated incidents")
    print("3. 👤 Examine highly connected policy holders (red borders)")
    print("4. 📮 Investigate high-density ZIP code areas")
    print("5. 🔴 Follow red edges (claims flagged as fraud)")
    print("6. 📊 Use dashboard insights for prioritized investigation")
    print()
    print("📋 This dashboard is optimized for fraud investigators and claims adjusters.")
    print("🎯 All fraud patterns are automatically highlighted and explained.")
    print()


def output_analysis_summary(graph, suspicious_nodes, suspicious_clusters, top_suspicious_nodes):
    """
    Print a comprehensive summary of the graph analysis results.

    Args:
        graph (nx.Graph): NetworkX graph object
        suspicious_nodes (dict): Dictionary containing suspicious nodes
        suspicious_clusters (dict): Dictionary containing suspicious clusters
        top_suspicious_nodes (list): List of top suspicious nodes
    """

    print("=" * 60)
    print("STEP 10: OUTPUT ANALYSIS SUMMARY")
    print("=" * 60)

    # Basic network statistics
    total_nodes = graph.number_of_nodes()
    total_edges = graph.number_of_edges()

    print("NETWORK STATISTICS:")
    print("-" * 30)
    print(f"Total nodes: {total_nodes}")
    print(f"Total edges: {total_edges}")
    print(f"Average degree: {2 * total_edges / total_nodes:.2f}")
    print(f"Network density: {nx.density(graph):.4f}")
    print()

    # Count suspicious entities
    total_suspicious_nodes = 0
    for metric in suspicious_nodes.values():
        for entity_type in metric.values():
            total_suspicious_nodes += len(entity_type)

    print("SUSPICIOUS ENTITIES DETECTED:")
    print("-" * 30)
    print(f"Total suspicious nodes: {total_suspicious_nodes}")

    for metric_name, metric_data in suspicious_nodes.items():
        print(f"\nBy {metric_name} centrality:")
        for entity_type, nodes in metric_data.items():
            print(f"  {entity_type}: {len(nodes)} nodes")

    print()

    # Cluster analysis
    total_clusters = len(suspicious_clusters['all_communities'])
    suspicious_cluster_count = len(suspicious_clusters['suspicious_clusters'])

    print("SUSPICIOUS CLUSTERS DETECTED:")
    print("-" * 30)
    print(f"Total communities: {total_clusters}")
    print(f"Suspicious clusters: {suspicious_cluster_count}")
    print(f"Cluster size threshold: {suspicious_clusters['size_threshold']}")

    if suspicious_clusters['suspicious_clusters']:
        largest_cluster = max(suspicious_clusters['suspicious_clusters'],
                            key=lambda x: x['size'])
        print(f"Largest suspicious cluster: {largest_cluster['size']} nodes")

    print()

    # Network connectivity analysis
    if nx.is_connected(graph):
        print("NETWORK CONNECTIVITY:")
        print("-" * 30)
        print("The network is fully connected.")
        print(f"Average shortest path length: {nx.average_shortest_path_length(graph):.2f}")
        print(f"Network diameter: {nx.diameter(graph)}")
    else:
        components = list(nx.connected_components(graph))
        print(f"NETWORK CONNECTIVITY:")
        print("-" * 30)
        print(f"The network has {len(components)} connected components.")
        largest_component_size = max(len(comp) for comp in components)
        print(f"Largest component size: {largest_component_size} nodes")

    print()

    # Final summary and recommendations
    print("ANALYSIS SUMMARY:")
    print("-" * 30)
    print("Graph-based analysis can uncover collusive fraud patterns that")
    print("traditional machine learning models may miss, including:")
    print()
    print("• Organized fraud rings with repeated entity connections")
    print("• Central players coordinating multiple fraudulent claims")
    print("• Geographic clusters of suspicious activity")
    print("• Vehicle sharing patterns across different policy holders")
    print("• Location-based fraud networks")
    print()
    print("RECOMMENDATIONS FOR INVESTIGATION:")
    print("-" * 30)
    print("1. Prioritize policy holders with highest centrality scores")
    print("2. Investigate vehicles appearing in multiple claims")
    print("3. Examine suspicious clusters for coordinated fraud")
    print("4. Cross-reference with external databases for entity verification")
    print("5. Monitor high-risk locations for future claims")
    print()

    return {
        'total_nodes': total_nodes,
        'total_edges': total_edges,
        'suspicious_nodes': total_suspicious_nodes,
        'top_suspicious_nodes': len(top_suspicious_nodes),
        'suspicious_clusters': suspicious_cluster_count
    }


def main():
    """
    Main function to execute the complete graph-based fraud detection analysis.
    """

    print("INSURANCE CLAIM FRAUD DETECTION USING GRAPH ANALYSIS")
    print("=" * 60)
    print()

    try:
        # Step 1: Load the feature-engineered dataset
        df = load_feature_engineered_dataset()

        # Step 2: Initialize the network graph
        G = initialize_network_graph()

        # Step 3: Create nodes for different entities with improved representation
        entity_nodes = create_nodes(G, df)

        # Step 4: Add edges based on relationships and compute shared entity scores
        add_edges_based_on_relationships(G, df)

        # Step 5: Compute graph metrics for centrality analysis
        centrality_metrics = compute_graph_metrics(G)

        # Step 6: Detect suspicious nodes and get top suspicious nodes for investigation
        suspicious_nodes, top_suspicious_nodes = detect_suspicious_nodes(G, centrality_metrics)

        # Step 7: Detect suspicious clusters using community detection
        suspicious_clusters = detect_suspicious_clusters(G)

        # Step 8: Print suspicious entities for investigation
        print_suspicious_entities(suspicious_nodes, suspicious_clusters)

        # Step 9: Create interactive investigation graph using PyVis
        create_interactive_investigation_graph(G, top_suspicious_nodes, centrality_metrics)

        # Step 10: Output comprehensive analysis summary
        summary = output_analysis_summary(G, suspicious_nodes, suspicious_clusters, top_suspicious_nodes)

        print("=" * 60)
        print("GRAPH-BASED FRAUD DETECTION ANALYSIS COMPLETED")
        print("=" * 60)
        print()
        print("Key findings:")
        print(f"• Analyzed {summary['total_nodes']} entities and {summary['total_edges']} relationships")
        print(f"• Identified {summary['suspicious_nodes']} suspicious entities")
        print(f"• Detected {summary['top_suspicious_nodes']} high-priority suspicious nodes")
        print(f"• Found {summary['suspicious_clusters']} potentially fraudulent clusters")
        print()
        print("Interactive investigation graph saved as: fraud_network_investigation.html")
        print("Open this file in a web browser to explore the fraud network interactively.")
        print()
        print("Next steps:")
        print("• Investigate high-priority suspicious entities in the interactive graph")
        print("• Cross-reference with additional data sources")
        print("• Monitor identified patterns in future claims")
        print("• Combine with traditional ML models for enhanced detection")

    except FileNotFoundError as e:
        print(f"Error: Dataset file not found - {e}")
        print("Please ensure the feature-engineered dataset exists at:")
        print("data/processed/insurance_claims_feature_engineered.csv")
    except Exception as e:
        print(f"Error during analysis: {e}")
        print("Please check the data and try again.")


# Execute the main function when the script is run
if __name__ == "__main__":
    main()

In [ ]:
"""
FRAUD INVESTIGATION VISUALIZATION SYSTEM
========================================

A comprehensive fraud detection and visualization system that creates
interactive investigation dashboards for insurance claims analysis.

This system identifies fraud patterns through graph analysis and provides
investigators with an intuitive interface for exploring suspicious relationships.

Author: Senior Software Engineer
Technology Stack: Python, NetworkX, PyVis, Pandas
"""

import pandas as pd
import networkx as nx
from pyvis.network import Network
from networkx.algorithms import community as nx_community
import numpy as np
import warnings
import webbrowser
import os
from typing import Dict, List, Tuple, Set, Any

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')


class FraudInvestigationSystem:
    """
    Main class for the fraud investigation visualization system.

    This system provides end-to-end functionality for:
    - Loading and processing insurance claims data
    - Building fraud detection graphs
    - Identifying suspicious patterns
    - Creating interactive investigation dashboards
    """

    def __init__(self):
        """Initialize the fraud investigation system."""
        self.graph = None
        self.dataset = None
        self.metrics = None
        self.suspicious_nodes = None
        self.clusters = None
        self.investigation_subgraph = None

    def load_dataset(self) -> pd.DataFrame:
        """
        Load the feature-engineered insurance claims dataset.

        Returns:
            pd.DataFrame: Loaded dataset with engineered features
        """
        print("=" * 80)
        print("STEP 1: LOADING FEATURE ENGINEERED DATASET")
        print("=" * 80)

        # Define the path to the feature-engineered dataset
        dataset_path = "data/processed/insurance_claims_feature_engineered.csv"

        try:
            # Load the dataset from CSV file
            self.dataset = pd.read_csv(dataset_path)

            # Display dataset summary
            print(f"✅ Dataset loaded successfully from: {dataset_path}")
            print(f"📊 Dataset shape: {self.dataset.shape}")
            print(f"📋 Number of claims: {len(self.dataset)}")
            print(f"🔢 Number of features: {self.dataset.shape[1]}")
            print()

            # Display fraud statistics
            fraud_count = self.dataset['fraud_reported'].sum()
            fraud_rate = (fraud_count / len(self.dataset)) * 100

            print(f"🚨 Fraud cases in dataset: {fraud_count}")
            print(f"📈 Fraud rate: {fraud_rate:.1f}%")
            print()

            print("🔍 Graph analysis is useful in fraud detection because:")
            print("  • Fraud rings often involve repeated connections between entities")
            print("  • Collusive behavior creates unusual network patterns")
            print("  • Traditional ML models may miss organized fraud networks")
            print("  • Graph metrics can identify central players in fraud schemes")
            print()

            return self.dataset

        except FileNotFoundError:
            print(f"❌ Error: Dataset file not found at {dataset_path}")
            print("Please ensure the feature-engineered dataset exists.")
            raise
        except Exception as e:
            print(f"❌ Error loading dataset: {e}")
            raise

    def initialize_graph(self) -> nx.Graph:
        """
        Initialize a NetworkX graph object for fraud detection.

        Returns:
            nx.Graph: Empty NetworkX graph object
        """
        print("=" * 80)
        print("STEP 2: INITIALIZING FRAUD DETECTION GRAPH")
        print("=" * 80)

        # Create an undirected graph for fraud analysis
        # Undirected because relationships are bidirectional in fraud networks
        self.graph = nx.Graph()

        print("✅ NetworkX graph initialized successfully.")
        print("📐 Graph type: Undirected")
        print()
        print("🏗️  Graph structure:")
        print("  • Nodes will represent entities:")
        print("    - Policy holders (policy_number)")
        print("    - Vehicles (auto_make + auto_model + auto_year)")
        print("    - Accident locations (incident_location)")
        print("    - Geographic areas (insured_zip)")
        print("  • Edges will represent relationships between entities")
        print("  • Fraud rings may involve repeated connections between entities")
        print()

        return self.graph

    def create_nodes(self, graph: nx.Graph, dataframe: pd.DataFrame) -> Dict[str, List[str]]:
        """
        Create nodes for different entities in the insurance claims data.

        Args:
            graph (nx.Graph): NetworkX graph object
            dataframe (pd.DataFrame): Insurance claims dataset

        Returns:
            Dict[str, List[str]]: Dictionary mapping entity types to their node identifiers
        """
        print("=" * 80)
        print("STEP 3: CREATING ENTITY NODES")
        print("=" * 80)

        # Initialize dictionary to store node identifiers by entity type
        entity_nodes = {
            'policy_holders': [],
            'vehicles': [],
            'locations': [],
            'zip_codes': []
        }

        # Create nodes for policy holders
        print("👤 Creating policy holder nodes...")
        for policy_num in dataframe['policy_number'].unique():
            node_id = f"policy_{policy_num}"
            graph.add_node(
                node_id,
                entity_type='policy_holder',
                policy_number=policy_num,
                shared_entity_score=0
            )
            entity_nodes['policy_holders'].append(node_id)

        print(f"  ✅ Created {len(entity_nodes['policy_holders'])} policy holder nodes")

        # Create nodes for vehicles with improved identifiers
        print("🚗 Creating vehicle nodes...")
        for _, row in dataframe.iterrows():
            # Create unique vehicle identifier with year for better distinction
            vehicle_id = f"vehicle_{row['auto_make']}_{row['auto_model']}_{row['auto_year']}"

            # Add node if it doesn't exist already
            if not graph.has_node(vehicle_id):
                graph.add_node(
                    vehicle_id,
                    entity_type='vehicle',
                    vehicle_make=row['auto_make'],
                    vehicle_model=row['auto_model'],
                    vehicle_year=row['auto_year'],
                    shared_entity_score=0
                )
                entity_nodes['vehicles'].append(vehicle_id)

        print(f"  ✅ Created {len(entity_nodes['vehicles'])} unique vehicle nodes")

        # Create nodes for incident locations
        print("📍 Creating location nodes...")
        for location in dataframe['incident_location'].unique():
            if pd.notna(location):  # Skip null values
                location_id = f"location_{location}"
                graph.add_node(
                    location_id,
                    entity_type='location',
                    incident_location=location,
                    shared_entity_score=0
                )
                entity_nodes['locations'].append(location_id)

        print(f"  ✅ Created {len(entity_nodes['locations'])} incident location nodes")

        # Create nodes for ZIP codes
        print("📮 Creating ZIP code nodes...")
        for zip_code in dataframe['insured_zip'].unique():
            if pd.notna(zip_code):  # Skip null values
                zip_id = f"zip_{zip_code}"
                graph.add_node(
                    zip_id,
                    entity_type='zip_code',
                    insured_zip=zip_code,
                    shared_entity_score=0
                )
                entity_nodes['zip_codes'].append(zip_id)

        print(f"  ✅ Created {len(entity_nodes['zip_codes'])} ZIP code nodes")
        print()

        # Summary
        total_nodes = graph.number_of_nodes()
        print(f"📊 Total nodes created: {total_nodes}")
        print()
        print("🏗️  Entity representation:")
        print("  • Vehicles: make_model_year (e.g., Toyota_Camry_2018)")
        print("  • Policy holders: policy_number")
        print("  • Locations: incident_location")
        print("  • ZIP codes: insured_zip")
        print()
        print("💾 Each node stores comprehensive attributes for fraud investigation.")
        print()

        return entity_nodes

    def create_edges(self, graph: nx.Graph, dataframe: pd.DataFrame) -> None:
        """
        Add edges to represent relationships between entities in claims.
        Also compute shared entity scores to identify potential fraud rings.

        Args:
            graph (nx.Graph): NetworkX graph object
            dataframe (pd.DataFrame): Insurance claims dataset
        """
        print("=" * 80)
        print("STEP 4: CREATING RELATIONSHIP EDGES")
        print("=" * 80)

        # Track entity usage for shared entity score computation
        vehicle_usage = {}
        location_usage = {}
        zip_usage = {}

        # Add edges between policy holders and vehicles
        print("🔗 Creating policy-to-vehicle edges...")
        vehicle_edge_count = 0
        for _, row in dataframe.iterrows():
            policy_id = f"policy_{row['policy_number']}"
            vehicle_id = f"vehicle_{row['auto_make']}_{row['auto_model']}_{row['auto_year']}"

            # Track vehicle usage for shared entity score
            if vehicle_id not in vehicle_usage:
                vehicle_usage[vehicle_id] = []
            vehicle_usage[vehicle_id].append(policy_id)

            # Add edge if both nodes exist
            if graph.has_node(policy_id) and graph.has_node(vehicle_id):
                graph.add_edge(
                    policy_id,
                    vehicle_id,
                    relationship_type='policy_vehicle',
                    claim_amount=row['total_claim_amount'],
                    fraud_flag=row['fraud_reported']
                )
                vehicle_edge_count += 1

        print(f"  ✅ Added {vehicle_edge_count} policy-to-vehicle edges")

        # Add edges between policy holders and incident locations
        print("📍 Creating policy-to-location edges...")
        location_edge_count = 0
        for _, row in dataframe.iterrows():
            policy_id = f"policy_{row['policy_number']}"

            # Skip if incident location is null
            if pd.isna(row['incident_location']):
                continue

            location_id = f"location_{row['incident_location']}"

            # Track location usage for shared entity score
            if location_id not in location_usage:
                location_usage[location_id] = []
            location_usage[location_id].append(policy_id)

            # Add edge if both nodes exist
            if graph.has_node(policy_id) and graph.has_node(location_id):
                graph.add_edge(
                    policy_id,
                    location_id,
                    relationship_type='policy_location',
                    claim_amount=row['total_claim_amount'],
                    fraud_flag=row['fraud_reported']
                )
                location_edge_count += 1

        print(f"  ✅ Added {location_edge_count} policy-to-location edges")

        # Add edges between policy holders and ZIP codes
        print("📮 Creating policy-to-ZIP edges...")
        zip_edge_count = 0
        for _, row in dataframe.iterrows():
            policy_id = f"policy_{row['policy_number']}"

            # Skip if ZIP code is null
            if pd.isna(row['insured_zip']):
                continue

            zip_id = f"zip_{row['insured_zip']}"

            # Track ZIP usage for shared entity score
            if zip_id not in zip_usage:
                zip_usage[zip_id] = []
            zip_usage[zip_id].append(policy_id)

            # Add edge if both nodes exist
            if graph.has_node(policy_id) and graph.has_node(zip_id):
                graph.add_edge(
                    policy_id,
                    zip_id,
                    relationship_type='policy_zip',
                    claim_amount=row['total_claim_amount'],
                    fraud_flag=row['fraud_reported']
                )
                zip_edge_count += 1

        print(f"  ✅ Added {zip_edge_count} policy-to-ZIP edges")
        print()

        # Compute shared entity scores for fraud ring detection
        print("🔍 Computing shared entity scores...")
        print("-" * 50)

        # Update shared entity scores for vehicles
        high_score_vehicles = 0
        for vehicle_id, policy_holders in vehicle_usage.items():
            if len(policy_holders) > 1:  # Vehicle used by multiple policy holders
                shared_score = len(policy_holders) - 1  # Score based on number of additional users
                if graph.has_node(vehicle_id):
                    graph.nodes[vehicle_id]['shared_entity_score'] = shared_score
                    high_score_vehicles += 1

        print(f"🚗 Vehicles shared by multiple policy holders: {high_score_vehicles}")

        # Update shared entity scores for locations
        high_score_locations = 0
        for location_id, policy_holders in location_usage.items():
            if len(policy_holders) > 1:  # Location used by multiple policy holders
                shared_score = len(policy_holders) - 1
                if graph.has_node(location_id):
                    graph.nodes[location_id]['shared_entity_score'] = shared_score
                    high_score_locations += 1

        print(f"📍 Locations shared by multiple policy holders: {high_score_locations}")

        # Update shared entity scores for ZIP codes
        high_score_zips = 0
        for zip_id, policy_holders in zip_usage.items():
            if len(policy_holders) > 1:  # ZIP connects multiple policy holders
                shared_score = len(policy_holders) - 1
                if graph.has_node(zip_id):
                    graph.nodes[zip_id]['shared_entity_score'] = shared_score
                    high_score_zips += 1

        print(f"📮 ZIP codes with multiple policy holders: {high_score_zips}")
        print()

        total_edges = graph.number_of_edges()
        print(f"📊 Total edges in graph: {total_edges}")
        print()
        print("💡 Shared entity scores help identify fraud rings where:")
        print("  • Multiple policy holders use the same vehicle")
        print("  • Multiple incidents occur at the same location")
        print("  • Many policy holders share the same geographic area")
        print()

    def compute_graph_metrics(self, graph: nx.Graph) -> Dict[str, Dict[str, float]]:
        """
        Compute centrality metrics to identify important nodes in the fraud network.

        Args:
            graph (nx.Graph): NetworkX graph object

        Returns:
            Dict[str, Dict[str, float]]: Dictionary containing centrality metrics for all nodes
        """
        print("=" * 80)
        print("STEP 5: COMPUTING GRAPH METRICS")
        print("=" * 80)

        # Compute degree centrality - measures how many connections each node has
        print("📊 Computing degree centrality...")
        degree_centrality = nx.degree_centrality(graph)

        # Compute betweenness centrality - measures how often a node appears on shortest paths
        print("🌉 Computing betweenness centrality...")
        betweenness_centrality = nx.betweenness_centrality(graph)

        # Compute eigenvector centrality - measures influence based on connections to influential nodes
        print("⭐ Computing eigenvector centrality...")
        try:
            eigenvector_centrality = nx.eigenvector_centrality(graph)
        except:
            # Handle cases where eigenvector centrality fails to converge
            eigenvector_centrality = {node: 0 for node in graph.nodes()}
            print("  ⚠️  Eigenvector centrality failed to converge, using zeros")

        print("✅ Graph metrics computed successfully:")
        print("  • Degree centrality: Connection frequency")
        print("  • Betweenness centrality: Bridge/broker importance")
        print("  • Eigenvector centrality: Influence in network")
        print()

        # Store all metrics in a dictionary
        self.metrics = {
            'degree': degree_centrality,
            'betweenness': betweenness_centrality,
            'eigenvector': eigenvector_centrality
        }

        print("🎯 Why these metrics help identify suspicious entities:")
        print("  High centrality nodes may indicate:")
        print("    • Frequently involved vehicles")
        print("    • Repeated incident locations")
        print("    • Policy holders linked to many claims")
        print("    • Central players in fraud networks")
        print()

        return self.metrics

    def detect_suspicious_entities(self, graph: nx.Graph, metrics: Dict[str, Dict[str, float]]) -> List[str]:
        """
        Identify nodes with unusually high centrality values.

        Args:
            graph (nx.Graph): NetworkX graph object
            metrics (Dict[str, Dict[str, float]]): Centrality metrics for all nodes

        Returns:
            List[str]: List of suspicious node identifiers
        """
        print("=" * 80)
        print("STEP 6: DETECTING SUSPICIOUS ENTITIES")
        print("=" * 80)

        # Define percentile threshold for identifying suspicious nodes (top 5%)
        percentile_threshold = 95
        suspicious_nodes = set()

        # Analyze degree centrality
        print("🔍 Analyzing degree centrality...")
        degree_values = list(metrics['degree'].values())
        degree_threshold = np.percentile(degree_values, percentile_threshold)
        print(f"  📊 Degree centrality threshold (95th percentile): {degree_threshold:.4f}")

        for node, centrality_value in metrics['degree'].items():
            if centrality_value >= degree_threshold:
                suspicious_nodes.add(node)

        # Analyze betweenness centrality
        print("🌉 Analyzing betweenness centrality...")
        betweenness_values = list(metrics['betweenness'].values())
        betweenness_threshold = np.percentile(betweenness_values, percentile_threshold)
        print(f"  📊 Betweenness centrality threshold (95th percentile): {betweenness_threshold:.4f}")

        for node, centrality_value in metrics['betweenness'].items():
            if centrality_value >= betweenness_threshold:
                suspicious_nodes.add(node)

        # Analyze eigenvector centrality
        print("⭐ Analyzing eigenvector centrality...")
        eigenvector_values = list(metrics['eigenvector'].values())
        eigenvector_threshold = np.percentile(eigenvector_values, percentile_threshold)
        print(f"  📊 Eigenvector centrality threshold (95th percentile): {eigenvector_threshold:.4f}")

        for node, centrality_value in metrics['eigenvector'].items():
            if centrality_value >= eigenvector_threshold:
                suspicious_nodes.add(node)

        # Convert to list for consistency
        self.suspicious_nodes = list(suspicious_nodes)

        print()
        print(f"🚨 Suspicious nodes identified based on high connectivity: {len(self.suspicious_nodes)} total")
        print("⚠️  Unusually high connectivity may indicate organized fraud rings.")
        print()

        return self.suspicious_nodes

    def detect_clusters(self, graph: nx.Graph) -> Dict[str, Any]:
        """
        Use community detection to identify tightly connected clusters.

        Args:
            graph (nx.Graph): NetworkX graph object

        Returns:
            Dict[str, Any]: Dictionary containing cluster information
        """
        print("=" * 80)
        print("STEP 7: DETECTING FRAUD CLUSTERS")
        print("=" * 80)

        # Use the Louvain algorithm for community detection
        print("🔍 Running Louvain community detection...")
        try:
            detected_communities = nx_community.louvain_communities(graph, seed=42)
        except:
            # Fallback to greedy modularity communities if Louvain fails
            print("  ⚠️  Louvain failed, using greedy modularity communities")
            detected_communities = nx_community.greedy_modularity_communities(graph)

        print(f"✅ Number of communities detected: {len(detected_communities)}")

        # Analyze community sizes
        community_sizes = [len(community) for community in detected_communities]
        print(f"📊 Community size range: {min(community_sizes)} to {max(community_sizes)}")
        print(f"📈 Average community size: {np.mean(community_sizes):.1f}")

        # Identify suspicious clusters (unusually large communities - top 10%)
        size_threshold = np.percentile(community_sizes, 90)
        suspicious_clusters = []

        print(f"🎯 Cluster size threshold (90th percentile): {size_threshold:.1f}")

        for i, community in enumerate(detected_communities):
            if len(community) >= size_threshold:
                # Analyze entity types in this community
                entity_types = {}
                for node in community:
                    node_attrs = graph.nodes[node]
                    entity_type = node_attrs.get('entity_type', 'unknown')
                    entity_types[entity_type] = entity_types.get(entity_type, 0) + 1

                suspicious_clusters.append({
                    'cluster_id': i,
                    'size': len(community),
                    'nodes': list(community),
                    'entity_types': entity_types
                })

        print(f"🚨 Number of suspicious clusters (top 10% largest): {len(suspicious_clusters)}")
        print()
        print("💡 How fraud rings may appear as tightly connected clusters:")
        print("  • Groups of policy holders sharing vehicles")
        print("  • Clusters of claims at same locations")
        print("  • Networks of entities in geographic proximity")
        print()

        # Store cluster information
        self.clusters = {
            'all_communities': list(detected_communities),
            'suspicious_clusters': suspicious_clusters,
            'size_threshold': size_threshold
        }

        return self.clusters

    def build_investigation_subgraph(self, graph: nx.Graph, suspicious_nodes: List[str], max_nodes: int = 60) -> nx.Graph:
        """
        Create a focused investigation subgraph for visualization.

        Args:
            graph (nx.Graph): Original NetworkX graph object
            suspicious_nodes (List[str]): List of suspicious node identifiers
            max_nodes (int): Maximum number of nodes for the investigation graph

        Returns:
            nx.Graph: Focused investigation subgraph
        """
        print("=" * 80)
        print("STEP 8: BUILDING INVESTIGATION SUBGRAPH")
        print("=" * 80)

        # Build investigation subgraph with top suspicious nodes and their neighbors
        nodes_to_include = set()

        # Take top suspicious nodes (prioritize by degree centrality)
        suspicious_with_centrality = []
        for node in suspicious_nodes:
            if node in self.metrics['degree']:
                suspicious_with_centrality.append((node, self.metrics['degree'][node]))

        # Sort by centrality and take top nodes
        suspicious_with_centrality.sort(key=lambda x: x[1], reverse=True)
        top_suspicious = [node for node, _ in suspicious_with_centrality[:20]]
        nodes_to_include.update(top_suspicious)

        # Add immediate neighbors of top suspicious nodes
        for suspicious_node in top_suspicious:
            if suspicious_node in graph.nodes():
                neighbors = list(graph.neighbors(suspicious_node))
                # Add up to 2 neighbors per suspicious node to keep size manageable
                nodes_to_include.update(neighbors[:2])

        # Limit total nodes to prevent clutter
        if len(nodes_to_include) > max_nodes:
            nodes_to_include = set(list(nodes_to_include)[:max_nodes])

        # Create investigation subgraph
        self.investigation_subgraph = graph.subgraph(nodes_to_include)

        print(f"🎯 Building investigation graph with {len(nodes_to_include)} nodes")
        print(f"📊 Including top suspicious nodes and their immediate connections")
        print(f"🔍 Focused view enables detailed fraud pattern analysis")
        print()

        return self.investigation_subgraph

    def create_interactive_investigation_graph(self, subgraph: nx.Graph) -> str:
        """
        Create an interactive fraud investigation graph using PyVis.

        Args:
            subgraph (nx.Graph): Investigation subgraph

        Returns:
            str: Path to the generated HTML file
        """
        print("=" * 80)
        print("STEP 9: CREATING INTERACTIVE INVESTIGATION DASHBOARD")
        print("=" * 80)

        # Initialize PyVis network with investigator-friendly configuration
        pyvis_network = Network(
            height='900px',
            width='1400px',
            bgcolor='#f8f9fa',
            font_color='black',
            notebook=False,
            cdn_resources='in_line',
            select_menu=True,
            filter_menu=True
        )

        # Configure physics for better layout using forceAtlas2Based
        physics_options = """
        var options = {
          "physics": {
            "enabled": true,
            "forceAtlas2Based": {
              "gravitationalConstant": -50,
              "centralGravity": 0.01,
              "springLength": 120,
              "springConstant": 0.08,
              "damping": 0.4,
              "avoidOverlap": 0.8
            },
            "minVelocity": 0.75,
            "solver": "forceAtlas2Based"
          },
          "interaction": {
            "hover": true,
            "tooltipDelay": 200,
            "zoomSpeed": 0.5,
            "dragNodes": true,
            "dragView": true,
            "zoomView": true
          },
          "layout": {
            "improvedLayout": true
          }
        }
        """
        pyvis_network.set_options(physics_options)

        # Identify suspicious nodes for highlighting
        suspicious_set = set(self.suspicious_nodes)

        # Add nodes with entity-specific visual design
        print("🎨 Adding nodes with custom visual design...")
        for node in subgraph.nodes():
            node_attrs = self.graph.nodes[node]
            entity_type = node_attrs.get('entity_type', 'unknown')

            # Determine node appearance based on entity type
            if entity_type == 'policy_holder':
                color = '#3498db'  # Blue
                shape = 'dot'
                policy_num = node_attrs.get('policy_number', 'Unknown')
                label = f"Policy {policy_num}"

            elif entity_type == 'vehicle':
                color = '#27ae60'  # Green
                shape = 'triangle'
                make = node_attrs.get('vehicle_make', 'Unknown')
                model = node_attrs.get('vehicle_model', 'Unknown')
                year = node_attrs.get('vehicle_year', 'Unknown')
                label = f"{make} {model}"

            elif entity_type == 'location':
                color = '#e74c3c'  # Red
                shape = 'square'
                location = node_attrs.get('incident_location', 'Unknown')
                label = f"Location {location}"

            elif entity_type == 'zip_code':
                color = '#f39c12'  # Orange
                shape = 'diamond'
                zip_code = node_attrs.get('insured_zip', 'Unknown')
                label = f"ZIP {zip_code}"

            else:
                color = '#95a5a6'  # Gray for unknown
                shape = 'dot'
                label = str(node)

            # Determine node size based on degree centrality
            degree_centrality = self.metrics['degree'].get(node, 0)
            base_size = 20
            node_size = base_size + (degree_centrality * 100)

            # Highlight high-risk nodes with red borders
            if node in suspicious_set:
                border_width = 3
                border_color = '#e74c3c'  # Red border for suspicious
            else:
                border_width = 1
                border_color = '#34495e'  # Dark gray border

            # Create investigation tooltip
            tooltip = self._create_investigation_tooltip(node, node_attrs, subgraph)

            # Add node to PyVis network
            pyvis_network.add_node(
                node,
                label=label,
                color=color,
                shape=shape,
                size=node_size,
                title=tooltip,
                borderWidth=border_width,
                borderColor=border_color,
                font={'size': 12, 'color': 'black'},
                chosen=True
            )

        # Add edges with fraud-specific styling
        print("🔗 Adding edges with fraud-specific styling...")
        for edge in subgraph.edges(data=True):
            source, target, edge_data = edge
            fraud_flag = edge_data.get('fraud_flag', 0)
            claim_amount = edge_data.get('claim_amount', 0)

            # Style edges based on fraud flag
            if fraud_flag:
                edge_color = '#e74c3c'  # Red for fraud
                edge_width = 3
                edge_title = f"Fraud Flagged Claim\nAmount: ${claim_amount:,.2f}"
            else:
                edge_color = '#95a5a6'  # Gray for normal
                edge_width = 1
                edge_title = f"Normal Claim\nAmount: ${claim_amount:,.2f}"

            pyvis_network.add_edge(
                source,
                target,
                color=edge_color,
                width=edge_width,
                title=edge_title,
                hoverWidth=3
            )

        # Generate investigation insights for dashboard
        insights = self._generate_investigation_insights(subgraph)

        # Create investigation dashboard HTML
        dashboard_html = self._create_dashboard_html(insights)

        # Generate HTML content and add dashboard
        html_content = pyvis_network.generate_html()

        # Insert dashboard after the body tag
        body_index = html_content.find('<body')
        if body_index != -1:
            body_end = html_content.find('>', body_index) + 1
            html_content = html_content[:body_end] + dashboard_html + html_content[body_end:]

        # Save HTML file
        output_file = "fraud_network_investigation.html"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html_content)

        print(f"✅ Interactive investigation dashboard saved as: {output_file}")
        print()

        # Auto-open in browser
        try:
            file_path = os.path.abspath(output_file)
            webbrowser.open(f'file://{file_path}')
            print(f"🌐 Investigation dashboard automatically opened in your browser")
        except Exception as e:
            print(f"⚠️  Could not auto-open browser. Please open {output_file} manually.")

        print()
        print("🎯 Dashboard Features:")
        print("  • Interactive graph exploration")
        print("  • Fraud pattern visualization")
        print("  • Investigation guidance")
        print("  • Entity relationship analysis")
        print()

        return output_file

    def _create_investigation_tooltip(self, node: str, node_attrs: Dict, subgraph: nx.Graph) -> str:
        """
        Create a detailed investigation tooltip for a node.

        Args:
            node (str): Node identifier
            node_attrs (Dict): Node attributes
            subgraph (nx.Graph): Investigation subgraph

        Returns:
            str: Formatted tooltip HTML
        """
        entity_type = node_attrs.get('entity_type', 'unknown')
        degree_centrality = self.metrics['degree'].get(node, 0)
        shared_score = node_attrs.get('shared_entity_score', 0)
        connections = subgraph.degree(node)

        # Determine risk level
        if degree_centrality > 0.002:
            risk_level = "HIGH"
            risk_emoji = "🚨"
        elif degree_centrality > 0.001:
            risk_level = "MEDIUM"
            risk_emoji = "⚠️"
        else:
            risk_level = "LOW"
            risk_emoji = "✅"

        # Entity-specific information
        if entity_type == 'policy_holder':
            policy_num = node_attrs.get('policy_number', 'Unknown')
            return f"""👤 POLICY HOLDER INVESTIGATION
═══════════════════════════════
Policy Number: {policy_num}
Connected Entities: {connections}
Network Influence: {degree_centrality:.4f}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• Check for unusual claim patterns
• Verify vehicle ownership legitimacy
• Cross-reference with incident locations
• High connectivity may indicate fraud ring involvement"""

        elif entity_type == 'vehicle':
            make = node_attrs.get('vehicle_make', 'Unknown')
            model = node_attrs.get('vehicle_model', 'Unknown')
            year = node_attrs.get('vehicle_year', 'Unknown')
            return f"""🚗 VEHICLE INVESTIGATION
═══════════════════════════════
Vehicle: {make} {model} ({year})
Number of Claims: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• Used by {connections} different policy holders
• High shared usage may indicate fraud ring
• Verify vehicle identification numbers
• Check for staged accidents"""

        elif entity_type == 'location':
            location = node_attrs.get('incident_location', 'Unknown')
            return f"""📍 LOCATION INVESTIGATION
═══════════════════════════════
Location: {location}
Number of Incidents: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• Multiple accidents at same location
• May indicate staged accident location
• Verify location legitimacy
• Check for repair shop connections"""

        elif entity_type == 'zip_code':
            zip_code = node_attrs.get('insured_zip', 'Unknown')
            return f"""📮 GEOGRAPHIC AREA INVESTIGATION
═══════════════════════════════
ZIP Code: {zip_code}
Policy Holders: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• High concentration of policy holders
• May indicate geographic fraud ring
• Cross-reference with incident locations
• Check for organized fraud patterns"""

        else:
            return f"Unknown Entity: {node}"

    def _generate_investigation_insights(self, subgraph: nx.Graph) -> Dict[str, int]:
        """
        Generate investigation insights for the dashboard.

        Args:
            subgraph (nx.Graph): Investigation subgraph

        Returns:
            Dict[str, int]: Dictionary of investigation metrics
        """
        suspicious_set = set(self.suspicious_nodes)

        total_policies = len([
            n for n in subgraph.nodes()
            if self.graph.nodes[n].get('entity_type') == 'policy_holder'
        ])

        suspicious_policies = len([
            n for n in subgraph.nodes()
            if n in suspicious_set and self.graph.nodes[n].get('entity_type') == 'policy_holder'
        ])

        repeated_vehicles = len([
            n for n in subgraph.nodes()
            if self.graph.nodes[n].get('entity_type') == 'vehicle' and subgraph.degree(n) > 2
        ])

        repeated_locations = len([
            n for n in subgraph.nodes()
            if self.graph.nodes[n].get('entity_type') == 'location' and subgraph.degree(n) > 2
        ])

        return {
            'total_policies': total_policies,
            'suspicious_policies': suspicious_policies,
            'repeated_vehicles': repeated_vehicles,
            'repeated_locations': repeated_locations
        }

    def _create_dashboard_html(self, insights: Dict[str, int]) -> str:
        """
        Create the investigation dashboard HTML.

        Args:
            insights (Dict[str, int]): Investigation insights

        Returns:
            str: Dashboard HTML content
        """
        return f"""
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 15px; margin: 10px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.3);">
    <h1 style="text-align: center; margin: 0 0 20px 0; font-size: 2.5em; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
        🔍 FRAUD INVESTIGATION DASHBOARD
    </h1>

    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
        <!-- Investigation Insights Panel -->
        <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
            <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">📊 INVESTIGATION INSIGHTS</h3>
            <div style="font-size: 0.95em; line-height: 1.6;">
                <div style="margin-bottom: 8px;"><strong>Total Policies Analyzed:</strong> {insights['total_policies']}</div>
                <div style="margin-bottom: 8px;"><strong>Suspicious Policies:</strong> <span style="color: #ff6b6b; font-weight: bold;">{insights['suspicious_policies']}</span></div>
                <div style="margin-bottom: 8px;"><strong>Repeated Vehicles Detected:</strong> <span style="color: #ffd93d; font-weight: bold;">{insights['repeated_vehicles']}</span></div>
                <div style="margin-bottom: 8px;"><strong>Repeated Locations Detected:</strong> <span style="color: #6bcf7f; font-weight: bold;">{insights['repeated_locations']}</span></div>
            </div>
        </div>

        <!-- Graph Legend Panel -->
        <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
            <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">📖 GRAPH LEGEND</h3>
            <div style="font-size: 0.9em; line-height: 1.5;">
                <div style="margin-bottom: 6px;"><span style="color: #3498db; font-weight: bold;">👤 Blue dots</span> → Policy Holders</div>
                <div style="margin-bottom: 6px;"><span style="color: #27ae60; font-weight: bold;">🚗 Green triangles</span> → Vehicles</div>
                <div style="margin-bottom: 6px;"><span style="color: #e74c3c; font-weight: bold;">📍 Red squares</span> → Accident Locations</div>
                <div style="margin-bottom: 6px;"><span style="color: #f39c12; font-weight: bold;">📮 Orange diamonds</span> → ZIP Codes</div>
                <div style="margin-bottom: 6px;"><span style="color: #e74c3c; font-weight: bold;">🔴 Red edges</span> → Fraud flagged claims</div>
                <div style="margin-bottom: 6px;"><span style="color: #95a5a6; font-weight: bold;">⚪ Gray edges</span> → Normal claims</div>
                <div style="margin-top: 8px; padding: 6px; background: rgba(255,255,255,0.2); border-radius: 5px;">
                    <strong>Large nodes</strong> = Frequently appearing<br>
                    <strong>Red borders</strong> = High fraud risk
                </div>
            </div>
        </div>
    </div>

    <!-- Investigation Explanation Panel -->
    <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
        <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">🧠 FRAUD PATTERN EXPLANATION</h3>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; font-size: 0.9em; line-height: 1.5;">
            <div>
                <h4 style="color: #6bcf7f; margin: 0 0 8px 0;">Understanding Fraud Patterns:</h4>
                <ul style="margin: 0; padding-left: 20px;">
                    <li><strong>Shared vehicle:</strong> Multiple policies connected to the same vehicle</li>
                    <li><strong>Repeated location:</strong> Many accidents at the same location</li>
                    <li><strong>High connectivity:</strong> Entities linked to many claims</li>
                    <li><strong>Fraud ring:</strong> Tightly connected cluster of claims</li>
                </ul>
            </div>
            <div>
                <h4 style="color: #6bcf7f; margin: 0 0 8px 0;">Investigation Actions:</h4>
                <ul style="margin: 0; padding-left: 20px;">
                    <li>🖱️ <strong>Hover</strong> for detailed investigation info</li>
                    <li>🔍 <strong>Zoom</strong> to examine specific areas</li>
                    <li>✋ <strong>Drag</strong> nodes to rearrange layout</li>
                    <li>🎯 <strong>Click</strong> to highlight connections</li>
                </ul>
            </div>
        </div>
    </div>

    <!-- Investigation Controls Panel -->
    <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px); margin-top: 15px;">
        <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">🛠 INVESTIGATION CONTROLS</h3>
        <div style="text-align: center;">
            <button onclick="network.setOptions({{physics:false}})" style="padding:10px 20px; margin:5px; border:none; border-radius:6px; background:#ffcc00; font-weight:bold; cursor:pointer;">
                ❄️ Freeze Layout
            </button>
            <button onclick="network.setOptions({{physics:true}})" style="padding:10px 20px; margin:5px; border:none; border-radius:6px; background:#28a745; color:white; font-weight:bold; cursor:pointer;">
                ⚡ Enable Physics
            </button>
            <button onclick="alert('🎯 INVESTIGATION GUIDE:\\n\\n1. Look for red-bordered nodes (high risk)\\n2. Follow red edges (fraud claims)\\n3. Find shared vehicles (green triangles)\\n4. Check repeated locations (red squares)\\n5. Identify clusters of connected entities\\n\\n💡 Use zoom and drag to explore patterns!')" style="padding:10px 20px; margin:5px; border:none; border-radius:6px; background:#007bff; color:white; font-weight:bold; cursor:pointer;">
                📋 Start Guided Tour
            </button>
        </div>
    </div>
</div>
"""

    def print_analysis_summary(self) -> Dict[str, Any]:
        """
        Print a comprehensive summary of the fraud analysis results.

        Returns:
            Dict[str, Any]: Summary statistics
        """
        print("=" * 80)
        print("STEP 10: ANALYSIS SUMMARY")
        print("=" * 80)

        # Basic network statistics
        total_nodes = self.graph.number_of_nodes()
        total_edges = self.graph.number_of_edges()

        print("📊 NETWORK STATISTICS:")
        print("-" * 40)
        print(f"Total nodes: {total_nodes}")
        print(f"Total edges: {total_edges}")
        print(f"Average degree: {2 * total_edges / total_nodes:.2f}")
        print(f"Network density: {nx.density(self.graph):.4f}")
        print()

        # Suspicious entities summary
        print("🚨 SUSPICIOUS ENTITIES DETECTED:")
        print("-" * 40)
        print(f"Total suspicious nodes: {len(self.suspicious_nodes)}")
        print()

        # Cluster analysis
        if self.clusters:
            total_clusters = len(self.clusters['all_communities'])
            suspicious_cluster_count = len(self.clusters['suspicious_clusters'])

            print("🔍 CLUSTER ANALYSIS:")
            print("-" * 40)
            print(f"Total communities: {total_clusters}")
            print(f"Suspicious clusters: {suspicious_cluster_count}")
            print()

        # Recommendations
        print("🎯 INVESTIGATION RECOMMENDATIONS:")
        print("-" * 40)
        print("1. Prioritize policy holders with highest centrality scores")
        print("2. Investigate vehicles appearing in multiple claims")
        print("3. Examine suspicious clusters for coordinated fraud")
        print("4. Cross-reference with external databases for verification")
        print("5. Monitor high-risk locations for future claims")
        print()

        return {
            'total_nodes': total_nodes,
            'total_edges': total_edges,
            'suspicious_nodes': len(self.suspicious_nodes),
            'suspicious_clusters': len(self.clusters['suspicious_clusters']) if self.clusters else 0
        }

    def run_complete_analysis(self) -> Dict[str, Any]:
        """
        Execute the complete fraud investigation analysis pipeline.

        Returns:
            Dict[str, Any]: Complete analysis results
        """
        print("🚀 FRAUD INVESTIGATION SYSTEM - COMPLETE ANALYSIS")
        print("=" * 80)
        print()

        try:
            # Step 1: Load dataset
            dataset = self.load_dataset()

            # Step 2: Initialize graph
            graph = self.initialize_graph()

            # Step 3: Create nodes
            entity_nodes = self.create_nodes(graph, dataset)

            # Step 4: Create edges
            self.create_edges(graph, dataset)

            # Step 5: Compute metrics
            metrics = self.compute_graph_metrics(graph)

            # Step 6: Detect suspicious entities
            suspicious_nodes = self.detect_suspicious_entities(graph, metrics)

            # Step 7: Detect clusters
            clusters = self.detect_clusters(graph)

            # Step 8: Build investigation subgraph
            investigation_subgraph = self.build_investigation_subgraph(graph, suspicious_nodes)

            # Step 9: Create interactive visualization
            dashboard_file = self.create_interactive_investigation_graph(investigation_subgraph)

            # Step 10: Print summary
            summary = self.print_analysis_summary()

            print("=" * 80)
            print("✅ FRAUD INVESTIGATION ANALYSIS COMPLETED")
            print("=" * 80)
            print()
            print("🎯 Key Findings:")
            print(f"  • Analyzed {summary['total_nodes']} entities and {summary['total_edges']} relationships")
            print(f"  • Identified {summary['suspicious_nodes']} suspicious entities")
            print(f"  • Found {summary['suspicious_clusters']} potentially fraudulent clusters")
            print()
            print("📊 Interactive dashboard saved as: fraud_network_investigation.html")
            print("🌐 Open this file in a web browser to explore the fraud network")
            print()
            print("🔍 Next Steps:")
            print("  • Investigate high-priority suspicious entities in the interactive graph")
            print("  • Cross-reference with additional data sources")
            print("  • Monitor identified patterns in future claims")
            print("  • Combine with traditional ML models for enhanced detection")

            return {
                'summary': summary,
                'dashboard_file': dashboard_file,
                'suspicious_nodes': suspicious_nodes,
                'clusters': clusters
            }

        except FileNotFoundError as e:
            print(f"❌ Error: Dataset file not found - {e}")
            print("Please ensure the feature-engineered dataset exists at:")
            print("data/processed/insurance_claims_feature_engineered.csv")
            raise
        except Exception as e:
            print(f"❌ Error during analysis: {e}")
            print("Please check the data and try again.")
            raise


def main():
    """
    Main function to execute the fraud investigation system.
    """
    # Initialize the fraud investigation system
    investigation_system = FraudInvestigationSystem()

    # Run the complete analysis
    try:
        results = investigation_system.run_complete_analysis()
        print("\n🎉 Fraud investigation completed successfully!")
        return results
    except Exception as e:
        print(f"\n❌ Fraud investigation failed: {e}")
        return None


if __name__ == "__main__":
    main()


In [ ]:
"""
Insurance Fraud Detection System - Clean Streamlit Application
================================================================

A clean, stable web interface for real-time fraud prediction using
native Streamlit components only.

Features:
- Real-time fraud risk assessment
- Clean, native Streamlit layout
- Color-coded risk levels
- Integration with decision_engine.py
- No fragile CSS hacks

Author: Insurance Claim Intelligence System Team
"""

import streamlit as st
import sys
from pathlib import Path

# Add src directory to path for imports
sys.path.append(str(Path(__file__).parent.parent / "src"))

try:
    from decision_engine import predict_claim_risk
except ImportError as e:
    st.error(f"Failed to import decision engine: {e}")
    st.stop()

# Configure page settings
st.set_page_config(
    page_title="Fraud Detection Bureau",
    page_icon="🕵️",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# Minimal safe CSS only
st.markdown("""
<style>
    /* Font styling only - safe */
    .main-header {
        font-family: 'Inter', sans-serif;
        font-size: 2.5rem;
        font-weight: 700;
        text-align: center;
        margin-bottom: 0.5rem;
    }

    .subtitle {
        font-family: 'Inter', sans-serif;
        text-align: center;
        font-size: 1rem;
        font-weight: 500;
        margin-bottom: 2rem;
        opacity: 0.8;
    }

    /* Button polish only - safe */
    .stButton > button {
        font-weight: 600;
        transition: all 0.2s ease;
    }

    /* Risk display enhancement */
    .risk-probability {
        font-size: 3.5rem;
        font-weight: 700;
        text-align: center;
        margin: 1rem 0;
    }

    .risk-level {
        font-size: 1.8rem;
        font-weight: 600;
        text-align: center;
        margin-bottom: 1rem;
    }
</style>
""", unsafe_allow_html=True)

def main():
    """Main application function - Professional Fraud Investigation Dashboard."""

    # Professional header
    st.markdown('<h1 class="main-header">🕵️ Fraud Detection Bureau</h1>', unsafe_allow_html=True)
    st.markdown('<p class="subtitle">Advanced Investigation System</p>', unsafe_allow_html=True)

    # System Status Dashboard
    st.subheader("📊 System Status")

    status_cols = st.columns(4)
    with status_cols[0]:
        st.metric("ML Engine", "🤖 ACTIVE")
    with status_cols[1]:
        st.metric("Rules System", "🔍 ARMED")
    with status_cols[2]:
        st.metric("Accuracy", "80.5%")
    with status_cols[3]:
        st.metric("Recall", "61.2%")

    # Initialize session state
    if 'prediction_done' not in st.session_state:
        st.session_state.prediction_done = False
        st.session_state.last_result = None
        st.session_state.last_input = None

    # Case Input Section
    st.subheader("🗂 Case File")

    with st.container():
        with st.form("fraud_detection_form"):
            # Financial Information Group
            st.write("**💰 Financial Information**")
            fin_cols = st.columns(2)
            with fin_cols[0]:
                total_claim_amount = st.text_input(
                    "Total Claim Amount ($)",
                    value="5000",
                    help="Enter the total claim amount"
                )

                annual_premium = st.text_input(
                    "Annual Premium ($)",
                    value="1000",
                    help="Enter the annual insurance premium"
                )

            with fin_cols[1]:
                injury_claim = st.text_input(
                    "Injury Claim Amount ($)",
                    value="0",
                    help="Enter injury claim amount"
                )

            st.write("**⚡ Incident Details**")
            incident_cols = st.columns(3)
            with incident_cols[0]:
                incident_severity = st.selectbox(
                    "Incident Severity",
                    options=["Minor Damage", "Major Damage", "Total Loss", "Trivial Damage"],
                    index=1
                )

            with incident_cols[1]:
                number_of_vehicles = st.text_input(
                    "Number of Vehicles",
                    value="1",
                    help="Number of vehicles involved"
                )

            with incident_cols[2]:
                property_damage = st.selectbox(
                    "Property Damage",
                    options=["NO", "YES"],
                    index=0
                )

            # Submit button
            submit_button = st.form_submit_button(
                "🔍 Open Investigation",
                use_container_width=True
            )

    # Process prediction when form is submitted
    if submit_button:
        # Input validation
        if not validate_inputs_text(total_claim_amount, annual_premium, injury_claim, number_of_vehicles):
            st.session_state.prediction_done = False
        else:
            with st.spinner("� Conducting forensic analysis..."):
                try:
                    # Convert text inputs to proper format
                    total_claim_amount_float = float(total_claim_amount)
                    annual_premium_float = float(annual_premium)
                    injury_claim_float = float(injury_claim)
                    number_of_vehicles_int = int(number_of_vehicles)

                    # Prepare input data for the model
                    input_data = prepare_input_data(
                        total_claim_amount_float,
                        annual_premium_float,
                        incident_severity,
                        number_of_vehicles_int,
                        injury_claim_float,
                        property_damage
                    )

                    # Get prediction from decision engine
                    result = predict_claim_risk(input_data)

                    # Apply improved thresholds
                    result = apply_improved_thresholds(result, input_data)

                    # Store in session state
                    st.session_state.prediction_done = True
                    st.session_state.last_result = result
                    st.session_state.last_input = input_data

                except Exception as e:
                    st.error(f"❌ Error during investigation: {str(e)}")
                    st.error("Please check your input values and try again.")
                    st.session_state.prediction_done = False

    # Display results ONLY after prediction is done
    if st.session_state.prediction_done and st.session_state.last_result:
        display_results(st.session_state.last_result, st.session_state.last_input)

def prepare_input_data(total_claim_amount, annual_premium, incident_severity,
                      number_of_vehicles_involved, injury_claim, property_damage):
    """
    Prepare input data in the format expected by the decision engine.

    Note: In production, this should match exactly the training feature set
    """

    # Basic feature engineering (simplified for demo)
    claim_to_premium_ratio = total_claim_amount / annual_premium if annual_premium > 0 else 0
    high_claim_indicator = 1 if total_claim_amount > 10000 else 0

    # Encode categorical variables (simplified encoding)
    incident_severity_encoded = {
        "Minor Damage": 0,
        "Major Damage": 1,
        "Severe Damage": 2,
        "Total Loss": 3
    }.get(incident_severity, 1)

    property_damage_encoded = 1 if property_damage == "YES" else 0

    # Create input dictionary with basic features
    # Note: In production, this should match exactly the training feature set
    input_data = {
        "total_claim_amount": total_claim_amount / 100000,  # Normalize
        "policy_annual_premium": annual_premium / 10000,   # Normalize
        "incident_severity": incident_severity_encoded,
        "number_of_vehicles_involved": number_of_vehicles_involved,
        "injury_claim": injury_claim / 100000,  # Normalize
        "property_damage": property_damage_encoded,
        "claim_to_premium_ratio": claim_to_premium_ratio,
        "high_claim_indicator": high_claim_indicator,

        # Add some default values for other required features
        "months_as_customer": 0.5,
        "age": 0.3,
        "policy_state": 0.0,
        "policy_csl": 0.06,
        "policy_deductable": -0.22,
        "umbrella_limit": -0.48,
        "insured_zip": -0.49,
        "insured_sex": 1.0,
        "capital-gains": 0.0,
        "capital-loss": 0.0,
        "incident_type": 0.0,
        "collision_type": 0.0,
        "authorities_contacted": 0.0,
        "incident_state": 0.0,
        "incident_hour_of_the_day": 0.5,
        "bodily_injuries": 0.0,
        "witnesses": 0.0,
        "police_report_available": 0.0,
        "property_claim": 0.0,
        "vehicle_claim": 0.0,
        "auto_year": 0.0,
        "injury_severity_score": 0.0,
        "vehicle_damage_flag": 0.0,
        "claim_to_vehicle_ratio": 0.0,
        "incident_hour_category": 0.0,
        "claim_component_sum": 0.0,
        "claim_breakdown_ratio": 0.0,
        "claim_breakdown_error": 0.0,
    }

    return input_data

def display_results(result, input_data):
    """Display real investigation workflow experience."""

    fraud_probability = result["fraud_probability"]
    risk_level = result["risk_level"]
    debug_info = result.get('debug_info', {})

    # Convert to percentage format
    fraud_prob_percent = fraud_probability * 100

    # 1. CASE FILE OPENED
    st.subheader("� Case File Opened")

    with st.container():
        st.write(f"**Case ID**: FD-{hash(str(input_data)) % 10000:04d}")
        st.write(f"**Claim Amount**: ${input_data.get('total_claim_amount', 0)*100000:,.0f}")
        st.write(f"**Incident Type**: {['Minor', 'Major', 'Severe', 'Total'][int(input_data.get('incident_severity', 1))]} Damage")
        st.write(f"**Vehicles Involved**: {int(input_data.get('number_of_vehicles_involved', 1))}")
        st.write(f"**Property Damage**: {'Yes' if input_data.get('property_damage', 0) > 0.5 else 'No'}")

    # 2. FORENSIC ANALYSIS
    st.subheader("🔬 Forensic Analysis")

    with st.container():
        # Probability display
        st.write(f"**Fraud Probability Score**: {fraud_prob_percent:.1f}%")

        # Rule triggers with proper interpretation
        triggered_rules = debug_info.get('triggered_rules', [])
        if triggered_rules:
            st.write("**Anomalies Detected**:")
            for rule in triggered_rules:
                if rule == "injury_exceeds_total":
                    if risk_level == "HIGH RISK":
                        st.write("🚨 Injury claim exceeds total claim amount")
                    elif risk_level == "MEDIUM RISK":
                        st.write("⚠️ Injury claim higher than total claim")
                    else:
                        st.write("ℹ️ Minor discrepancy in injury vs total claim")

                elif rule == "excessive_vehicles":
                    if risk_level == "HIGH RISK":
                        st.write("🚨 Unusual number of vehicles involved")
                    elif risk_level == "MEDIUM RISK":
                        st.write("⚠️ Multiple vehicles in incident")
                    else:
                        st.write("ℹ️ Multiple vehicles noted")

                elif rule == "suspicious_ratio_high":
                    if risk_level == "HIGH RISK":
                        st.write("🚨 Extremely high claim-to-premium ratio")
                    elif risk_level == "MEDIUM RISK":
                        st.write("⚠️ Elevated claim-to-premium ratio")
                    else:
                        st.write("ℹ️ Above-average claim ratio")

                elif rule == "suspicious_ratio_med":
                    if risk_level == "MEDIUM RISK":
                        st.write("⚠️ High claim-to-premium ratio")
                    else:
                        st.write("ℹ️ Slightly elevated claim ratio")

                elif rule == "severe_incident":
                    if risk_level == "HIGH RISK":
                        st.write("🚨 Total loss incident detected")
                    elif risk_level == "MEDIUM RISK":
                        st.write("⚠️ High severity incident")
                    else:
                        st.write("ℹ️ Severe incident type")

            if len(triggered_rules) >= 3:
                if risk_level == "HIGH RISK":
                    st.write("🚨 Multiple suspicious patterns detected")
                elif risk_level == "MEDIUM RISK":
                    st.write("⚠️ Several anomalies noted")
                else:
                    st.write("ℹ️ Minor anomalies detected but not sufficient for fraud classification")
        else:
            st.write("**No anomalies detected in claim analysis**")

    # 3. FINDINGS
    st.subheader("📊 Findings")

    with st.container():
        explanation = generate_human_readable_explanation(fraud_probability, input_data, debug_info)
        for point in explanation:
            st.write(f"• {point}")

    # 4. DECISION
    st.subheader("⚖️ Decision")

    with st.container():
        if risk_level == "HIGH RISK":
            st.markdown(f'<div class="risk-probability" style="color: #dc2626;">{fraud_prob_percent:.1f}%</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="risk-level" style="color: #dc2626;">🚨 FRAUD ALERT</div>', unsafe_allow_html=True)
            st.error("**This case exhibits strong indicators of fraudulent activity**")

        elif risk_level == "MEDIUM RISK":
            st.markdown(f'<div class="risk-probability" style="color: #d97706;">{fraud_prob_percent:.1f}%</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="risk-level" style="color: #d97706;">⚠️ REVIEW REQUIRED</div>', unsafe_allow_html=True)
            st.warning("**This case requires enhanced review before processing**")

        else:
            st.markdown(f'<div class="risk-probability" style="color: #16a34a;">{fraud_prob_percent:.1f}%</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="risk-level" style="color: #16a34a;">✅ APPROVED</div>', unsafe_allow_html=True)
            st.success("**This case is cleared for standard processing**")

    # 5. CASE STATUS (MUST ADD)
    st.subheader("🔒 Case Status")

    with st.container():
        if risk_level == "HIGH RISK":
            st.error("🔴 **CASE ESCALATED – FRAUD SUSPECTED**")
            st.write("- Case forwarded to Fraud Investigation Division")
            st.write("- Payment processing suspended pending review")
            st.write("- Policyholder flagged for enhanced monitoring")
            st.write("- Additional documentation required")

        elif risk_level == "MEDIUM RISK":
            st.warning("🟡 **CASE FLAGGED – REVIEW REQUIRED**")
            st.write("- Case assigned to senior claims analyst")
            st.write("- Additional verification steps required")
            st.write("- Standard processing delayed until review complete")
            st.write("- Monitor for additional red flags")

        else:
            st.success("🟢 **CASE CLOSED – NO FRAUD DETECTED**")
            st.write("- Claim approved for standard processing")
            st.write("- Payment processing authorized")
            st.write("- Case filed as routine claim")
            st.write("- No further action required")

    # Technical Details (expander)
    with st.expander("🔧 Technical Analysis"):
        display_model_info()

        if debug_info:
            st.write("**Technical Details:**")
            st.write(f"- **Raw ML Probability**: {debug_info.get('raw_probability', 0)*100:.2f}%")
            st.write(f"- **Rule-Based Boost**: +{debug_info.get('rule_boost', 0)*100:.1f}%")
            st.write(f"- **Adjusted Probability**: {debug_info.get('adjusted_probability', 0)*100:.2f}%")
            st.write(f"- **Claim-to-Premium Ratio**: {debug_info.get('claim_to_premium_ratio', 0):.1f}x")
            st.write(f"- **Rules Triggered**: {len(debug_info.get('triggered_rules', []))}")

def validate_inputs_text(total_claim_amount, annual_premium, injury_claim, number_of_vehicles):
    """Validate text inputs with clean error messages."""
    errors = []

    # Check if inputs are numeric
    try:
        claim_amount = float(total_claim_amount)
        if claim_amount <= 0:
            errors.append("Claim amount must be greater than $0")
        if claim_amount > 1000000:
            errors.append("Claim amount exceeds reasonable limit ($1,000,000)")
    except ValueError:
        errors.append("Claim amount must be a valid number")

    try:
        premium = float(annual_premium)
        if premium <= 0:
            errors.append("Annual premium must be greater than $0")
    except ValueError:
        errors.append("Annual premium must be a valid number")

    try:
        injury = float(injury_claim)
        if injury < 0:
            errors.append("Injury claim cannot be negative")
    except ValueError:
        errors.append("Injury claim must be a valid number")

    try:
        vehicles = int(number_of_vehicles)
        if vehicles <= 0:
            errors.append("Number of vehicles must be greater than 0")
        if vehicles > 10:
            errors.append("Number of vehicles exceeds reasonable limit (10)")
    except ValueError:
        errors.append("Number of vehicles must be a valid integer")

    if errors:
        for error in errors:
            st.error(f"❌ {error}")
        return False
    return True

def apply_improved_thresholds(result, input_data):
    """Apply improved thresholds with rule-based overrides for fraud detection."""
    raw_probability = result["fraud_probability"]

    # Extract actual values for rule-based analysis
    claim_amount = input_data.get('total_claim_amount', 0) * 100000
    annual_premium = input_data.get('policy_annual_premium', 0) * 10000
    injury_claim = input_data.get('injury_claim', 0) * 100000
    vehicles = input_data.get('number_of_vehicles_involved', 1)
    severity = input_data.get('incident_severity', 1)

    # Compute correct claim-to-premium ratio
    claim_to_premium_ratio = claim_amount / annual_premium if annual_premium > 0 else 0

    # DEBUG OUTPUT
    print(f"\n=== FRAUD DETECTION DEBUG ===")
    print(f"Raw ML Probability: {raw_probability:.4f}")

    # RULE-BASED OVERRIDE SYSTEM
    triggered_rules = []
    rule_boost = 0

    # Rule 1: Injury claim exceeds total claim (impossible)
    if injury_claim > claim_amount and injury_claim > 0:
        triggered_rules.append("injury_exceeds_total")
        rule_boost += 0.25
        print(f"RULE TRIGGERED: Injury claim (${injury_claim:,.0f}) > Total claim (${claim_amount:,.0f})")

    # Rule 2: Excessive vehicles
    if vehicles > 3:
        triggered_rules.append("excessive_vehicles")
        rule_boost += 0.20
        print(f"RULE TRIGGERED: Excessive vehicles ({vehicles})")

    # Rule 3: Suspicious claim-to-premium ratio
    if claim_to_premium_ratio > 5:
        triggered_rules.append("suspicious_ratio_high")
        rule_boost += 0.30
        print(f"RULE TRIGGERED: Very high claim-to-premium ratio ({claim_to_premium_ratio:.1f})")
    elif claim_to_premium_ratio > 3:
        triggered_rules.append("suspicious_ratio_med")
        rule_boost += 0.15
        print(f"RULE TRIGGERED: High claim-to-premium ratio ({claim_to_premium_ratio:.1f})")

    # Rule 4: Severe incident
    if severity >= 3:  # Total Loss
        triggered_rules.append("severe_incident")
        rule_boost += 0.10
        print(f"RULE TRIGGERED: Severe incident (Total Loss)")

    # ANOMALY BOOSTING for multiple signals
    if len(triggered_rules) >= 3:
        anomaly_boost = 0.15
        rule_boost += anomaly_boost
        print(f"ANOMALY BOOST: +{anomaly_boost:.2f} (multiple suspicious signals)")

    # Calculate adjusted probability
    adjusted_probability = min(raw_probability + rule_boost, 1.0)
    print(f"Rule Boost: +{rule_boost:.4f}")
    print(f"Adjusted Probability: {adjusted_probability:.4f}")

    # Apply NEW tighter thresholds
    if adjusted_probability >= 0.60:
        risk_level = "HIGH RISK"
    elif adjusted_probability >= 0.30:
        risk_level = "MEDIUM RISK"
    else:
        risk_level = "LOW RISK"

    print(f"Final Risk Level: {risk_level}")
    print(f"Triggered Rules: {triggered_rules}")
    print("========================\n")

    # Store debug info for explanations
    debug_info = {
        'raw_probability': raw_probability,
        'adjusted_probability': adjusted_probability,
        'triggered_rules': triggered_rules,
        'claim_to_premium_ratio': claim_to_premium_ratio,
        'rule_boost': rule_boost
    }

    return {
        "fraud_probability": adjusted_probability,
        "risk_level": risk_level,
        "debug_info": debug_info
    }

def generate_human_readable_explanation(fraud_probability, input_data, debug_info=None):
    """Generate human-readable explanations without contradictions."""
    explanations = []

    # Extract actual values
    claim_amount = input_data.get('total_claim_amount', 0) * 100000
    annual_premium = input_data.get('policy_annual_premium', 0) * 10000
    injury_claim = input_data.get('injury_claim', 0) * 100000
    vehicles = input_data.get('number_of_vehicles_involved', 1)
    severity = input_data.get('incident_severity', 1)

    # Use debug info if available for accurate explanations
    if debug_info:
        triggered_rules = debug_info.get('triggered_rules', [])
        claim_to_premium_ratio = debug_info.get('claim_to_premium_ratio', 0)

        # Explain triggered rules only (no contradictions)
        if "injury_exceeds_total" in triggered_rules:
            explanations.append("Injury claim exceeds total claim amount - impossible scenario")

        if "excessive_vehicles" in triggered_rules:
            explanations.append(f"Unusual number of vehicles involved ({vehicles})")

        if "suspicious_ratio_high" in triggered_rules:
            explanations.append(f"Claim amount is {claim_to_premium_ratio:.1f}x the annual premium - extremely suspicious")
        elif "suspicious_ratio_med" in triggered_rules:
            explanations.append(f"Claim amount is {claim_to_premium_ratio:.1f}x the annual premium - unusually high")

        if "severe_incident" in triggered_rules:
            explanations.append("Total loss incident indicates high claim complexity")

        if len(triggered_rules) >= 3:
            explanations.append("Multiple suspicious patterns detected simultaneously")

    # Fallback explanations if no rules triggered
    if not explanations:
        if claim_amount > 50000:
            explanations.append("High claim amount requires additional review")
        elif claim_amount < 5000:
            explanations.append("Low claim amount appears normal")
        else:
            explanations.append("Claim amount is within typical range")

        if vehicles > 2:
            explanations.append("Multiple vehicles increase claim complexity")
        else:
            explanations.append("Single vehicle incident - standard complexity")

        severity_map = {0: "Minor", 1: "Major", 2: "Severe", 3: "Total Loss"}
        severity_text = severity_map.get(int(severity), "Unknown")
        if severity >= 2:
            explanations.append(f"High incident severity ({severity_text})")
        else:
            explanations.append(f"Standard incident severity ({severity_text})")

    # Overall assessment based on probability
    if fraud_probability > 0.7:
        explanations.append("Strong fraud indicators detected")
    elif fraud_probability > 0.4:
        explanations.append("Some suspicious characteristics noted")
    else:
        explanations.append("Minimal fraud indicators detected")

    return explanations[:4]  # Return top 4 explanations

def display_model_info():
    """Display model information in the expander."""
    st.write("**Model Performance Metrics**")
    model_cols = st.columns(3)

    with model_cols[0]:
        st.metric("Accuracy", "80.5%")
    with model_cols[1]:
        st.metric("Recall", "61.2%")
    with model_cols[2]:
        st.metric("F1 Score", "60.6%")

    st.write("**Model Details**")
    st.write("- **Model Type**: XGBoost Classifier")
    st.write("- **Training Data**: 1,000 insurance claims")
    st.write("- **Features**: 46 engineered features")
    st.write("- **Class Imbalance Handling**: Yes (scale_pos_weight)")

    st.write("**Risk Thresholds**")
    st.write("- **HIGH RISK**: Probability ≥ 60%")
    st.write("- **MEDIUM RISK**: Probability ≥ 30%")
    st.write("- **LOW RISK**: Probability < 30%")

    st.write("**Rule-Based Overrides**")
    st.write("- Injury > Total Claim: +25% probability")
    st.write("- Vehicles > 3: +20% probability")
    st.write("- Claim/Premium > 5: +30% probability")
    st.write("- Claim/Premium > 3: +15% probability")
    st.write("- Severe Incident: +10% probability")
    st.write("- Multiple Rules: +15% anomaly boost")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'streamlit'

In [ ]:
"""
FRAUD INVESTIGATION VISUALIZATION SYSTEM
========================================

A comprehensive fraud detection and visualization system that creates
interactive investigation dashboards for insurance claims analysis.

This system identifies fraud patterns through graph analysis and provides
investigators with an intuitive interface for exploring suspicious relationships.

Author: Senior Software Engineer
Technology Stack: Python, NetworkX, PyVis, Pandas
"""

import pandas as pd
import networkx as nx
try:
    # PyVis is used to generate the interactive HTML network visualization.
    from pyvis.network import Network
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Missing dependency: 'pyvis'. Install it in your environment with: pip install pyvis"
    ) from e
from networkx.algorithms import community as nx_community
import numpy as np
import warnings
import webbrowser
import os
import sys
from typing import Dict, List, Tuple, Set, Any

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Ensure console printing does not fail on Windows terminals with limited encodings.
# We prefer UTF-8 output, and fall back to replacing unsupported characters.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")


class FraudInvestigationSystem:
    """
    Main class for the fraud investigation visualization system.

    This system provides end-to-end functionality for:
    - Loading and processing insurance claims data
    - Building fraud detection graphs
    - Identifying suspicious patterns
    - Creating interactive investigation dashboards
    """

    def __init__(self):
        """Initialize the fraud investigation system."""
        self.graph = None
        self.dataset = None
        self.metrics = None
        self.suspicious_nodes = None
        self.clusters = None
        self.investigation_subgraph = None

    def load_dataset(self) -> pd.DataFrame:
        """
        Load the graph-based insurance claims dataset for visualization.

        Returns:
            pd.DataFrame: Loaded dataset with graph entity identifiers
        """
        print("=" * 80)
        print("STEP 1: LOADING GRAPH DATASET FOR VISUALIZATION")
        print("=" * 80)

        # Define the path to the graph dataset
        dataset_path = "data/processed/insurance_claims_graph_dataset.csv"

        try:
            # Load the dataset from CSV file
            self.dataset = pd.read_csv(dataset_path)

            # Display dataset summary
            print(f"✅ Graph dataset loaded successfully from: {dataset_path}")
            print(f"📊 Dataset shape: {self.dataset.shape}")
            print(f"📋 Number of claims: {len(self.dataset)}")
            print(f"🔢 Number of features: {self.dataset.shape[1]}")
            print()

            # Display graph entity statistics
            print("🏗️  Graph Entity Statistics:")
            print(f"  • Unique policies: {self.dataset['policy_id'].nunique()}")
            print(f"  • Unique vehicles: {self.dataset['vehicle_id'].nunique()}")
            print(f"  • Unique locations: {self.dataset['location_id'].nunique()}")
            print(f"  • Unique ZIP codes: {self.dataset['zip_id'].nunique()}")
            print()

            # Display fraud statistics (handle NaN values)
            fraud_data = self.dataset['fraud_reported'].dropna()
            if len(fraud_data) > 0:
                fraud_count = fraud_data.sum()
                fraud_rate = (fraud_count / len(fraud_data)) * 100
                print(f"🚨 Fraud cases in dataset: {fraud_count}")
                print(f"📈 Fraud rate: {fraud_rate:.1f}%")
            else:
                print("⚠️  No fraud data available in dataset")
            print()

            print("🔍 Graph analysis is useful in fraud detection because:")
            print("  • Fraud rings often involve repeated connections between entities")
            print("  • Collusive behavior creates unusual network patterns")
            print("  • Traditional ML models may miss organized fraud networks")
            print("  • Graph metrics can identify central players in fraud schemes")
            print()

            return self.dataset

        except FileNotFoundError:
            print(f"❌ Error: Dataset file not found at {dataset_path}")
            print("Please ensure the feature-engineered dataset exists.")
            raise
        except Exception as e:
            print(f"❌ Error loading dataset: {e}")
            raise

    def initialize_graph(self) -> nx.Graph:
        """
        Initialize a NetworkX graph object for fraud detection.

        Returns:
            nx.Graph: Empty NetworkX graph object
        """
        print("=" * 80)
        print("STEP 2: INITIALIZING FRAUD DETECTION GRAPH")
        print("=" * 80)

        # Create an undirected graph for fraud analysis
        # Undirected because relationships are bidirectional in fraud networks
        self.graph = nx.Graph()

        print("✅ NetworkX graph initialized successfully.")
        print("📐 Graph type: Undirected")
        print()
        print("🏗️  Graph structure:")
        print("  • Nodes will represent entities:")
        print("    - Policy holders (policy_id)")
        print("    - Vehicles (vehicle_id)")
        print("    - Accident locations (location_id)")
        print("    - Geographic areas (zip_id)")
        print("  • Edges will represent relationships between entities")
        print("  • Real-world entity names preserved for investigation")
        print()

        return self.graph

    def create_nodes(self, graph: nx.Graph, dataframe: pd.DataFrame) -> Dict[str, List[str]]:
        """
        Create nodes for different entities in the insurance claims data.

        Args:
            graph (nx.Graph): NetworkX graph object
            dataframe (pd.DataFrame): Insurance claims dataset

        Returns:
            Dict[str, List[str]]: Dictionary mapping entity types to their node identifiers
        """
        print("=" * 80)
        print("STEP 3: CREATING ENTITY NODES")
        print("=" * 80)

        # Initialize dictionary to store node identifiers by entity type
        entity_nodes = {
            "policy_holders": [],
            "vehicles": [],
            "locations": [],
            "zip_codes": [],
        }

        # Identify confirmed fraud policies so we can color their nodes red.
        # A policy is considered fraud if it has at least one claim with fraud_reported == 1.
        # Cast IDs to string so they are safe to use in both NetworkX and PyVis.
        fraud_policy_ids = set(
            str(x)
            for x in dataframe.loc[dataframe["fraud_reported"] == 1, "policy_id"].dropna().unique().tolist()
        )

        # -------------------------------------------------------------------
        # Policy holder nodes (node_id = policy_id)
        # Visualization rules:
        #   - color = blue (default)
        #   - label = policy_id
        #   - if fraud_reported == 1 for that policy, color = red
        # -------------------------------------------------------------------
        print("👤 Creating policy holder nodes...")
        for policy_id_raw in dataframe["policy_id"].dropna().unique():
            policy_id = str(policy_id_raw)
            node_id = f"policy:{policy_id}"
            base_color = "#3498db"  # Blue
            is_fraud_policy = policy_id in fraud_policy_ids
            color = "#e74c3c" if is_fraud_policy else base_color  # Red for confirmed fraud policies

            graph.add_node(
                node_id,
                entity_type="policy_holder",
                policy_id=policy_id,
                label=str(policy_id),
                base_color=base_color,
                color=color,
                fraud_policy=is_fraud_policy,
            )
            entity_nodes["policy_holders"].append(node_id)

        print(f"  ✅ Created {len(entity_nodes['policy_holders'])} policy holder nodes")

        # -------------------------------------------------------------------
        # Vehicle nodes (node_id = vehicle_id)
        # Visualization rule:
        #   - color = orange
        #   - label = auto_make + auto_model + auto_year
        # -------------------------------------------------------------------
        print("🚗 Creating vehicle nodes...")
        vehicle_rows = dataframe.dropna(subset=["vehicle_id"]).drop_duplicates(subset=["vehicle_id"])
        for _, row in vehicle_rows.iterrows():
            vehicle_id = str(row["vehicle_id"])
            node_id = f"vehicle:{vehicle_id}"

            make = str(row.get("auto_make", "")).strip()
            model = str(row.get("auto_model", "")).strip()
            year = str(row.get("auto_year", "")).strip()
            vehicle_label = " ".join([x for x in [make, model, year] if x and x != "nan"]).strip()
            if not vehicle_label:
                vehicle_label = str(vehicle_id)

            graph.add_node(
                node_id,
                entity_type="vehicle",
                vehicle_id=vehicle_id,
                auto_make=row.get("auto_make"),
                auto_model=row.get("auto_model"),
                auto_year=row.get("auto_year"),
                label=vehicle_label,
                base_color="#f39c12",  # Orange
                color="#f39c12",
            )
            entity_nodes["vehicles"].append(node_id)

        print(f"  ✅ Created {len(entity_nodes['vehicles'])} vehicle nodes")

        # -------------------------------------------------------------------
        # Location nodes (node_id = location_id)
        # Visualization rule:
        #   - color = green
        #   - label = incident_city + incident_state
        # -------------------------------------------------------------------
        print("📍 Creating location nodes...")
        location_rows = dataframe.dropna(subset=["location_id"]).drop_duplicates(subset=["location_id"])
        for _, row in location_rows.iterrows():
            location_id = str(row["location_id"])
            node_id = f"location:{location_id}"

            city = str(row.get("incident_city", "")).strip()
            state = str(row.get("incident_state", "")).strip()
            location_label = " ".join([x for x in [city, state] if x and x != "nan"]).strip()
            if not location_label:
                location_label = str(location_id)

            graph.add_node(
                node_id,
                entity_type="location",
                location_id=location_id,
                incident_city=row.get("incident_city"),
                incident_state=row.get("incident_state"),
                label=location_label,
                base_color="#27ae60",  # Green
                color="#27ae60",
            )
            entity_nodes["locations"].append(node_id)

        print(f"  ✅ Created {len(entity_nodes['locations'])} location nodes")

        # -------------------------------------------------------------------
        # ZIP code nodes (node_id = zip_id)
        # Visualization rule:
        #   - color = purple
        #   - label = insured_zip
        # -------------------------------------------------------------------
        print("📮 Creating ZIP code nodes...")
        zip_rows = dataframe.dropna(subset=["zip_id"]).drop_duplicates(subset=["zip_id"])
        for _, row in zip_rows.iterrows():
            zip_id = str(row["zip_id"])
            node_id = f"zip:{zip_id}"

            insured_zip = row.get("insured_zip")
            zip_label = str(insured_zip) if pd.notna(insured_zip) else str(zip_id)

            graph.add_node(
                node_id,
                entity_type="zip_code",
                zip_id=zip_id,
                insured_zip=insured_zip,
                label=zip_label,
                base_color="#9b59b6",  # Purple
                color="#9b59b6",
            )
            entity_nodes["zip_codes"].append(node_id)

        print(f"  ✅ Created {len(entity_nodes['zip_codes'])} ZIP code nodes")
        print()

        print(f"📊 Total nodes created: {graph.number_of_nodes()}")
        print()

        return entity_nodes

    def create_edges(self, graph: nx.Graph, dataframe: pd.DataFrame) -> None:
        """
        Add edges to represent relationships between entities in claims.
        Also compute shared entity scores to identify potential fraud rings.

        Args:
            graph (nx.Graph): NetworkX graph object
            dataframe (pd.DataFrame): Insurance claims dataset
        """
        print("=" * 80)
        print("STEP 4: CREATING RELATIONSHIP EDGES")
        print("=" * 80)

        # Track entity usage for shared entity score computation
        vehicle_usage = {}
        location_usage = {}
        zip_usage = {}

        # Add edges between policy holders and vehicles
        print("🔗 Creating policy-to-vehicle edges...")
        vehicle_edge_count = 0
        for _, row in dataframe.iterrows():
            policy_id = row['policy_id']
            vehicle_id = row['vehicle_id']

            # Skip if either ID is null
            if pd.isna(policy_id) or pd.isna(vehicle_id):
                continue

            # Cast node IDs to string for compatibility with PyVis
            policy_id = str(policy_id)
            vehicle_id = str(vehicle_id)

            # Use namespaced node IDs to avoid collisions between entity types
            policy_node = f"policy:{policy_id}"
            vehicle_node = f"vehicle:{vehicle_id}"

            # Track vehicle usage for shared entity score
            if vehicle_id not in vehicle_usage:
                vehicle_usage[vehicle_id] = []
            vehicle_usage[vehicle_id].append(policy_id)

            # Add edge if both nodes exist
            if graph.has_node(policy_node) and graph.has_node(vehicle_node):
                graph.add_edge(
                    policy_node,
                    vehicle_node,
                    relationship_type='policy_vehicle',
                    claim_amount=row['total_claim_amount'],
                    fraud_flag=row['fraud_reported']
                )
                vehicle_edge_count += 1

        print(f"  ✅ Added {vehicle_edge_count} policy-to-vehicle edges")

        # Add edges between policy holders and incident locations
        print("📍 Creating policy-to-location edges...")
        location_edge_count = 0
        for _, row in dataframe.iterrows():
            policy_id = row['policy_id']
            location_id = row['location_id']

            # Skip if either ID is null
            if pd.isna(policy_id) or pd.isna(location_id):
                continue

            # Cast node IDs to string for compatibility with PyVis
            policy_id = str(policy_id)
            location_id = str(location_id)

            # Use namespaced node IDs to avoid collisions between entity types
            policy_node = f"policy:{policy_id}"
            location_node = f"location:{location_id}"

            # Track location usage for shared entity score
            if location_id not in location_usage:
                location_usage[location_id] = []
            location_usage[location_id].append(policy_id)

            # Add edge if both nodes exist
            if graph.has_node(policy_node) and graph.has_node(location_node):
                graph.add_edge(
                    policy_node,
                    location_node,
                    relationship_type='policy_location',
                    claim_amount=row['total_claim_amount'],
                    fraud_flag=row['fraud_reported']
                )
                location_edge_count += 1

        print(f"  ✅ Added {location_edge_count} policy-to-location edges")

        # Add edges between policy holders and ZIP codes
        print("📮 Creating policy-to-ZIP edges...")
        zip_edge_count = 0
        for _, row in dataframe.iterrows():
            policy_id = row['policy_id']
            zip_id = row['zip_id']

            # Skip if either ID is null
            if pd.isna(policy_id) or pd.isna(zip_id):
                continue

            # Cast node IDs to string for compatibility with PyVis
            policy_id = str(policy_id)
            zip_id = str(zip_id)

            # Use namespaced node IDs to avoid collisions between entity types
            policy_node = f"policy:{policy_id}"
            zip_node = f"zip:{zip_id}"

            # Track ZIP usage for shared entity score
            if zip_id not in zip_usage:
                zip_usage[zip_id] = []
            zip_usage[zip_id].append(policy_id)

            # Add edge if both nodes exist
            if graph.has_node(policy_node) and graph.has_node(zip_node):
                graph.add_edge(
                    policy_node,
                    zip_node,
                    relationship_type='policy_zip',
                    claim_amount=row['total_claim_amount'],
                    fraud_flag=row['fraud_reported']
                )
                zip_edge_count += 1

        print(f"  ✅ Added {zip_edge_count} policy-to-ZIP edges")
        print()

        # Compute shared entity scores for fraud ring detection
        print("🔍 Computing shared entity scores...")
        print("-" * 50)

        # Update shared entity scores for vehicles
        high_score_vehicles = 0
        for vehicle_id, policy_holders in vehicle_usage.items():
            if len(policy_holders) > 1:  # Vehicle used by multiple policy holders
                shared_score = len(policy_holders) - 1  # Score based on number of additional users
                vehicle_node = f"vehicle:{vehicle_id}"
                if graph.has_node(vehicle_node):
                    graph.nodes[vehicle_node]['shared_entity_score'] = shared_score
                    high_score_vehicles += 1

        print(f"🚗 Vehicles shared by multiple policy holders: {high_score_vehicles}")

        # Update shared entity scores for locations
        high_score_locations = 0
        for location_id, policy_holders in location_usage.items():
            if len(policy_holders) > 1:  # Location used by multiple policy holders
                shared_score = len(policy_holders) - 1
                location_node = f"location:{location_id}"
                if graph.has_node(location_node):
                    graph.nodes[location_node]['shared_entity_score'] = shared_score
                    high_score_locations += 1

        print(f"📍 Locations shared by multiple policy holders: {high_score_locations}")

        # Update shared entity scores for ZIP codes
        high_score_zips = 0
        for zip_id, policy_holders in zip_usage.items():
            if len(policy_holders) > 1:  # ZIP connects multiple policy holders
                shared_score = len(policy_holders) - 1
                zip_node = f"zip:{zip_id}"
                if graph.has_node(zip_node):
                    graph.nodes[zip_node]['shared_entity_score'] = shared_score
                    high_score_zips += 1

        print(f"📮 ZIP codes with multiple policy holders: {high_score_zips}")
        print()

        total_edges = graph.number_of_edges()
        print(f"📊 Total edges in graph: {total_edges}")
        print()
        print("💡 Shared entity scores help identify fraud rings where:")
        print("  • Multiple policy holders use the same vehicle")
        print("  • Multiple incidents occur at the same location")
        print("  • Many policy holders share the same geographic area")
        print()

    def compute_graph_metrics(self, graph: nx.Graph) -> Dict[str, Dict[str, float]]:
        """
        Compute centrality metrics to identify important nodes in the fraud network.

        Args:
            graph (nx.Graph): NetworkX graph object

        Returns:
            Dict[str, Dict[str, float]]: Dictionary containing centrality metrics for all nodes
        """
        print("=" * 80)
        print("STEP 5: COMPUTING GRAPH METRICS")
        print("=" * 80)

        # Degree centrality: measures how connected each node is in the network.
        print("📊 Computing degree centrality...")
        degree_centrality = nx.degree_centrality(graph)

        # Betweenness centrality: measures how often a node sits on shortest paths.
        print("🌉 Computing betweenness centrality...")
        betweenness_centrality = nx.betweenness_centrality(graph)

        # Store only the required metrics for investigation tooltips and suspicious detection.
        self.metrics = {
            "degree": degree_centrality,
            "betweenness": betweenness_centrality,
        }

        print("✅ Graph metrics computed successfully.")
        print()

        return self.metrics

    def detect_suspicious_entities(self, graph: nx.Graph, metrics: Dict[str, Dict[str, float]]) -> List[str]:
        """
        Identify nodes with unusually high centrality values.

        Args:
            graph (nx.Graph): NetworkX graph object
            metrics (Dict[str, Dict[str, float]]): Centrality metrics for all nodes

        Returns:
            List[str]: List of suspicious node identifiers
        """
        print("=" * 80)
        print("STEP 6: DETECTING SUSPICIOUS ENTITIES")
        print("=" * 80)

        # Requirement: suspicious nodes are those with degree_centrality > 0.05.
        threshold = 0.05
        print(f"🔍 Flagging nodes with degree_centrality > {threshold:.2f} ...")

        suspicious_nodes = [node for node, value in metrics["degree"].items() if value > threshold]

        # Store for visualization highlighting (yellow).
        self.suspicious_nodes = suspicious_nodes

        print(f"🚨 Suspicious nodes detected: {len(self.suspicious_nodes)}")
        print()

        return self.suspicious_nodes

    def detect_clusters(self, graph: nx.Graph) -> Dict[str, Any]:
        """
        Use community detection to identify tightly connected clusters.

        Args:
            graph (nx.Graph): NetworkX graph object

        Returns:
            Dict[str, Any]: Dictionary containing cluster information
        """
        print("=" * 80)
        print("STEP 7: DETECTING FRAUD CLUSTERS")
        print("=" * 80)

        # Use the Louvain algorithm for community detection
        print("🔍 Running Louvain community detection...")
        try:
            detected_communities = nx_community.louvain_communities(graph, seed=42)
        except:
            # Fallback to greedy modularity communities if Louvain fails
            print("  ⚠️  Louvain failed, using greedy modularity communities")
            detected_communities = nx_community.greedy_modularity_communities(graph)

        print(f"✅ Number of communities detected: {len(detected_communities)}")

        # Analyze community sizes
        community_sizes = [len(community) for community in detected_communities]
        print(f"📊 Community size range: {min(community_sizes)} to {max(community_sizes)}")
        print(f"📈 Average community size: {np.mean(community_sizes):.1f}")

        # Identify suspicious clusters (unusually large communities - top 10%)
        size_threshold = np.percentile(community_sizes, 90)
        suspicious_clusters = []

        print(f"🎯 Cluster size threshold (90th percentile): {size_threshold:.1f}")

        for i, community in enumerate(detected_communities):
            if len(community) >= size_threshold:
                # Analyze entity types in this community
                entity_types = {}
                for node in community:
                    node_attrs = graph.nodes[node]
                    entity_type = node_attrs.get('entity_type', 'unknown')
                    entity_types[entity_type] = entity_types.get(entity_type, 0) + 1

                suspicious_clusters.append({
                    'cluster_id': i,
                    'size': len(community),
                    'nodes': list(community),
                    'entity_types': entity_types
                })

        print(f"🚨 Number of suspicious clusters (top 10% largest): {len(suspicious_clusters)}")
        print()
        print("💡 How fraud rings may appear as tightly connected clusters:")
        print("  • Groups of policy holders sharing vehicles")
        print("  • Clusters of claims at same locations")
        print("  • Networks of entities in geographic proximity")
        print()

        # Store cluster information
        self.clusters = {
            'all_communities': list(detected_communities),
            'suspicious_clusters': suspicious_clusters,
            'size_threshold': size_threshold
        }

        return self.clusters

    def build_investigation_subgraph(self, graph: nx.Graph, suspicious_nodes: List[str], max_nodes: int = 60) -> nx.Graph:
        """
        Create a focused investigation subgraph for visualization.

        Args:
            graph (nx.Graph): Original NetworkX graph object
            suspicious_nodes (List[str]): List of suspicious node identifiers
            max_nodes (int): Maximum number of nodes for the investigation graph

        Returns:
            nx.Graph: Focused investigation subgraph
        """
        print("=" * 80)
        print("STEP 8: BUILDING INVESTIGATION SUBGRAPH")
        print("=" * 80)

        # Build investigation subgraph with top suspicious nodes and their neighbors
        nodes_to_include = set()

        # Take top suspicious nodes (prioritize by degree centrality)
        suspicious_with_centrality = []
        for node in suspicious_nodes:
            if node in self.metrics['degree']:
                suspicious_with_centrality.append((node, self.metrics['degree'][node]))

        # Sort by centrality and take top nodes
        suspicious_with_centrality.sort(key=lambda x: x[1], reverse=True)
        top_suspicious = [node for node, _ in suspicious_with_centrality[:20]]
        nodes_to_include.update(top_suspicious)

        # Add immediate neighbors of top suspicious nodes
        for suspicious_node in top_suspicious:
            if suspicious_node in graph.nodes():
                neighbors = list(graph.neighbors(suspicious_node))
                # Add up to 2 neighbors per suspicious node to keep size manageable
                nodes_to_include.update(neighbors[:2])

        # Limit total nodes to prevent clutter
        if len(nodes_to_include) > max_nodes:
            nodes_to_include = set(list(nodes_to_include)[:max_nodes])

        # Create investigation subgraph
        self.investigation_subgraph = graph.subgraph(nodes_to_include)

        print(f"🎯 Building investigation graph with {len(nodes_to_include)} nodes")
        print(f"📊 Including top suspicious nodes and their immediate connections")
        print(f"🔍 Focused view enables detailed fraud pattern analysis")
        print()

        return self.investigation_subgraph

    def create_interactive_investigation_graph(self, subgraph: nx.Graph) -> str:
        """
        Create an interactive fraud investigation graph using PyVis.

        Args:
            subgraph (nx.Graph): Investigation graph (full graph or focused subgraph)

        Returns:
            str: Path to the generated HTML file
        """
        print("=" * 80)
        print("STEP 9: CREATING INTERACTIVE FRAUD INVESTIGATION NETWORK")
        print("=" * 80)

        # Initialize PyVis network (we will inject our own investigator UI controls,
        # so we disable the default PyVis menus).
        pyvis_network = Network(
            height='900px',
            width='100%',
            bgcolor='#ffffff',
            font_color='black',
            notebook=False,
            cdn_resources='in_line',
            select_menu=False,
            filter_menu=False
        )

        # Configure physics: use Barnes-Hut to spread clusters apart and reduce overlap.
        physics_options = """
        var options = {
          "physics": {
            "enabled": true,
            "solver": "barnesHut",
            "barnesHut": {
              "gravitationalConstant": -2000,
              "centralGravity": 0.3,
              "springLength": 120
            }
          },
          "interaction": {
            "hover": true,
            "tooltipDelay": 200,
            "zoomSpeed": 0.5,
            "dragNodes": true,
            "dragView": true,
            "zoomView": true
          },
          "layout": {
            "improvedLayout": true
          }
        }
        """
        pyvis_network.set_options(physics_options)

        # Identify suspicious nodes (degree_centrality > 0.05) for investigator highlighting.
        suspicious_set = set(self.suspicious_nodes or [])

        # Pre-compute investigator filter sets.
        # These sets are exported to the browser so investigators can filter interactively.
        fraud_edge_keys: Set[Tuple[str, str]] = set()
        fraud_connected_nodes: Set[str] = set()
        shared_vehicle_nodes: Set[str] = set()
        frequent_location_nodes: Set[str] = set()

        # Fraud edges / fraud-connected nodes (fraud_reported == 1)
        for u, v, data in subgraph.edges(data=True):
            if int(data.get("fraud_flag", 0)) == 1:
                fraud_edge_keys.add(tuple(sorted([u, v])))
                fraud_connected_nodes.add(u)
                fraud_connected_nodes.add(v)

        # Shared vehicles: vehicle nodes used by multiple policies (shared_entity_score > 0)
        for node, attrs in subgraph.nodes(data=True):
            if attrs.get("entity_type") == "vehicle" and float(attrs.get("shared_entity_score", 0)) > 0:
                shared_vehicle_nodes.add(node)

        # Frequent accident locations: locations with more than N incidents.
        # An incident corresponds to a policy-location relationship, so degree() is an incident count.
        for node, attrs in subgraph.nodes(data=True):
            if attrs.get("entity_type") == "location":
                incident_count = subgraph.degree(node)
                # Store on node for the tooltip and client-side filtering.
                attrs["incident_count"] = incident_count
                if incident_count > 5:  # Default N for initial view; UI lets investigators change this.
                    frequent_location_nodes.add(node)

        # Add nodes with required visualization rules:
        #   - Shapes by entity type
        #   - Sizes by degree centrality
        #   - Colors by entity type with fraud policy override
        #   - Suspicious highlighting by yellow border (NOT fill)
        print("🎨 Adding nodes with required visualization rules...")
        for node in subgraph.nodes():
            node_attrs = subgraph.nodes[node]
            entity_type = node_attrs.get("entity_type", "unknown")

            # Centrality metrics for tooltips
            degree_centrality = self.metrics["degree"].get(node, 0.0)
            betweenness_centrality = self.metrics["betweenness"].get(node, 0.0)

            # Base color is stored on the node during node creation (blue/orange/green/purple)
            base_color = node_attrs.get("base_color", "#95a5a6")

            # Apply fraud highlighting: confirmed fraud policy nodes are red.
            if entity_type == "policy_holder" and node_attrs.get("fraud_policy", False):
                color = "#e74c3c"  # Red
            else:
                color = base_color

            # Node label is stored on the node during node creation.
            label = node_attrs.get("label", str(node))

            # Shapes by entity type (investigator-friendly).
            if entity_type == "policy_holder":
                shape = "dot"       # circle
            elif entity_type == "vehicle":
                shape = "triangle"
            elif entity_type == "location":
                shape = "square"
            elif entity_type == "zip_code":
                shape = "diamond"
            else:
                shape = "dot"

            # Extract the human-readable entity identifier for tooltips.
            if entity_type == "policy_holder":
                entity_id = node_attrs.get("policy_id", "")
            elif entity_type == "vehicle":
                entity_id = node_attrs.get("vehicle_id", "")
            elif entity_type == "location":
                entity_id = node_attrs.get("location_id", "")
            elif entity_type == "zip_code":
                entity_id = node_attrs.get("zip_id", "")
            else:
                entity_id = ""

            # Node size reflects importance: size = 12 + degree_centrality * 200 (capped at 45).
            node_size = 12 + (degree_centrality * 200)
            node_size = min(node_size, 45)

            # Shared entity score is computed during edge creation (e.g., reused vehicle/location/zip).
            shared_score = float(node_attrs.get("shared_entity_score", 0))

            # Number of connections = node degree in the graph.
            connections = subgraph.degree(node)

            # Tooltip shows the required investigation fields for analysts.
            tooltip = (
                f"<b>Entity Type:</b> {entity_type}<br>"
                f"<b>Entity ID:</b> {entity_id}<br>"
                f"<b>Label:</b> {label}<br>"
                f"<b>Connections:</b> {connections}<br>"
                f"<b>Degree centrality:</b> {degree_centrality:.4f}<br>"
                f"<b>Betweenness centrality:</b> {betweenness_centrality:.4f}<br>"
                f"<b>Shared entity score:</b> {shared_score:.0f}"
            )

            # Suspicious nodes (degree_centrality > 0.05) get a yellow border and thicker outline.
            if node in suspicious_set:
                border_width = 4
                border_color = "#f1c40f"  # Yellow border
            else:
                border_width = 1
                border_color = "#2c3e50"  # Dark outline

            pyvis_network.add_node(
                node,
                label=label,
                color=color,
                size=node_size,
                title=tooltip,
                shape=shape,
                borderWidth=border_width,
                borderWidthSelected=border_width,
                borderColor=border_color,
                # Export attributes for client-side investigator filters.
                entity_type=entity_type,
                entity_id=str(entity_id),
                is_fraud_policy=bool(node_attrs.get("fraud_policy", False)),
                is_suspicious=bool(node in suspicious_set),
                is_shared_vehicle=bool(node in shared_vehicle_nodes),
                is_frequent_location=bool(node in frequent_location_nodes),
                incident_count=int(node_attrs.get("incident_count", 0)),
                shared_entity_score=float(node_attrs.get("shared_entity_score", 0)),
                degree_centrality=float(degree_centrality),
                betweenness_centrality=float(betweenness_centrality),
                font={"size": 12, "color": "black"},
            )

        # Add edges with fraud-specific styling
        print("🔗 Adding edges with fraud-specific styling...")
        for source, target, edge_data in subgraph.edges(data=True):
            fraud_flag = edge_data.get('fraud_flag', 0)
            claim_amount = edge_data.get('claim_amount', 0)

            # Edge styling rules:
            #   - Fraud edges: red, width 3
            #   - Normal edges: light grey, width 1
            if fraud_flag:
                edge_color = '#e74c3c'
                edge_width = 3
            else:
                edge_color = '#d0d3d4'
                edge_width = 1

            # Edge tooltip for investigators (supports the high-claim filter).
            edge_title = f"Claim Amount: ${claim_amount:,.2f}<br>Fraud Flag: {int(fraud_flag)}"

            pyvis_network.add_edge(
                source,
                target,
                color=edge_color,
                width=edge_width,
                title=edge_title,
                # Export edge attributes for client-side filters.
                fraud_flag=int(fraud_flag),
                claim_amount=float(claim_amount),
                hoverWidth=3
            )

        # Generate HTML content
        html_content = pyvis_network.generate_html()

        # ---------------------------------------------------------------
        # Structured UI layout (Title / Left panel / Center graph / Legend).
        # We inject CSS + HTML + JavaScript controls into the PyVis HTML.
        # ---------------------------------------------------------------

        # Title (top)
        title_html = """
<div id="icins-title" style="
  position: fixed;
  left: 0;
  right: 0;
  top: 0;
  height: 56px;
  z-index: 9999;
  display: flex;
  align-items: center;
  padding: 0 16px;
  background: #0b1320;
  color: #ffffff;
  font-family: 'Segoe UI', Arial, sans-serif;
  font-size: 18px;
  font-weight: 700;
  box-shadow: 0 2px 10px rgba(0,0,0,0.2);
">
  Insurance Fraud Investigation Network
</div>
"""

        # Investigation filter panel (left)
        panel_html = f"""
<div id="icins-panel" style="
  position: fixed;
  left: 12px;
  top: 72px;
  bottom: 160px;
  width: 320px;
  z-index: 9999;
  background: rgba(255, 255, 255, 0.98);
  border: 1px solid #e5e7eb;
  border-radius: 12px;
  padding: 14px 14px 10px 14px;
  font-family: 'Segoe UI', Arial, sans-serif;
  font-size: 13px;
  overflow: auto;
  box-shadow: 0 6px 24px rgba(0,0,0,0.10);
">
  <div style="font-weight:700; font-size:14px; margin-bottom:10px;">Investigation Filters</div>

  <label style="display:block; margin: 8px 0;">
    <input type="checkbox" id="f_showFraud" /> <b>Show Fraud Cases</b><br>
    <span style="color:#6b7280;">Display only nodes connected to fraud_reported = 1.</span>
  </label>

  <label style="display:block; margin: 8px 0;">
    <input type="checkbox" id="f_sharedVehicles" /> <b>Shared Vehicles</b><br>
    <span style="color:#6b7280;">Vehicles used by multiple policies (possible rings).</span>
  </label>

  <label style="display:block; margin: 8px 0;">
    <input type="checkbox" id="f_suspiciousNodes" /> <b>Suspicious Nodes</b><br>
    <span style="color:#6b7280;">degree_centrality &gt; </span>
    <input type="number" id="f_degreeThreshold" value="0.05" step="0.01" min="0" max="1"
           style="width:90px; padding:4px; margin-left:6px; border:1px solid #e5e7eb; border-radius:6px;" />
  </label>

  <div style="margin: 12px 0 6px 0;"><b>Frequent Accident Locations</b></div>
  <div style="color:#6b7280; margin-bottom:6px;">Locations with more than N incidents.</div>
  <div style="display:flex; gap:8px; align-items:center; margin-bottom:10px;">
    <input type="checkbox" id="f_frequentLocations" />
    <span>N &gt;</span>
    <input type="number" id="f_locationN" value="5" step="1" min="1" max="9999"
           style="width:90px; padding:4px; border:1px solid #e5e7eb; border-radius:6px;" />
  </div>

  <div style="margin: 12px 0 6px 0;"><b>High Claim Amount Connections</b></div>
  <div style="color:#6b7280; margin-bottom:6px;">Edges where claim_amount &gt; threshold.</div>
  <div style="display:flex; gap:8px; align-items:center;">
    <input type="checkbox" id="f_highClaimEdges" />
    <span>$</span>
    <input type="number" id="f_claimThreshold" value="50000" step="5000" min="0"
           style="width:120px; padding:4px; border:1px solid #e5e7eb; border-radius:6px;" />
  </div>

  <hr style="margin: 14px 0; border: none; border-top: 1px solid #eef2f7;" />

  <div style="margin-bottom:8px;"><b>Maximum nodes to display</b></div>
  <input type="range" id="f_nodeLimit" min="10" max="{max(10, subgraph.number_of_nodes())}" value="{min(500, subgraph.number_of_nodes())}" step="10"
         style="width: 100%;" />
  <div style="display:flex; justify-content:space-between; color:#6b7280; margin-top:6px;">
    <span>10</span>
    <span id="f_nodeLimitValue">{min(500, subgraph.number_of_nodes())}</span>
    <span>{subgraph.number_of_nodes()}</span>
  </div>

  <div style="margin-top: 12px; display:flex; gap:10px;">
    <button id="btn_applyFilters" style="
      flex:1;
      padding:10px 12px;
      border:none;
      border-radius:10px;
      background:#2563eb;
      color:white;
      font-weight:700;
      cursor:pointer;
    ">Apply filters</button>
    <button id="btn_resetFilters" style="
      padding:10px 12px;
      border:1px solid #e5e7eb;
      border-radius:10px;
      background:white;
      font-weight:700;
      cursor:pointer;
    ">Reset</button>
  </div>
</div>
"""

        # Legend (bottom-left, not overlapping the panel controls)
        legend_html = """
<div id="icins-legend" style="
  position: fixed;
  left: 12px;
  bottom: 12px;
  width: 320px;
  z-index: 9999;
  background: rgba(255, 255, 255, 0.98);
  border: 1px solid #e5e7eb;
  border-radius: 12px;
  padding: 12px 14px;
  font-family: 'Segoe UI', Arial, sans-serif;
  font-size: 13px;
  box-shadow: 0 6px 24px rgba(0,0,0,0.10);
">
  <div style="font-weight: 700; margin-bottom: 8px;">Legend</div>
  <div style="display:flex; align-items:center; margin:4px 0;">
    <span style="display:inline-block;width:12px;height:12px;background:#3498db;border-radius:50%;margin-right:8px;"></span>
    Blue = Policy Holder
  </div>
  <div style="display:flex; align-items:center; margin:4px 0;">
    <span style="display:inline-block;width:12px;height:12px;background:#f39c12;border-radius:2px;transform: rotate(0deg);margin-right:8px;"></span>
    Orange = Vehicle
  </div>
  <div style="display:flex; align-items:center; margin:4px 0;">
    <span style="display:inline-block;width:12px;height:12px;background:#27ae60;border-radius:2px;margin-right:8px;"></span>
    Green = Location
  </div>
  <div style="display:flex; align-items:center; margin:4px 0;">
    <span style="display:inline-block;width:12px;height:12px;background:#9b59b6;border-radius:2px;margin-right:8px;"></span>
    Purple = ZIP Code
  </div>
  <div style="display:flex; align-items:center; margin:4px 0;">
    <span style="display:inline-block;width:12px;height:12px;background:#e74c3c;border-radius:50%;margin-right:8px;"></span>
    Red = Confirmed Fraud
  </div>
  <div style="display:flex; align-items:center; margin:4px 0;">
    <span style="display:inline-block;width:12px;height:12px;border:3px solid #f1c40f;border-radius:50%;margin-right:8px;"></span>
    Yellow Border = Suspicious Node
  </div>
</div>
"""

        # CSS to shift the graph canvas to the center area (title + left panel)
        css_html = """
<style>
  body { margin: 0; padding: 0; overflow: hidden; }
  #mynetwork {
    position: fixed !important;
    top: 56px !important;
    left: 356px !important;
    right: 0 !important;
    bottom: 0 !important;
    width: auto !important;
    height: auto !important;
    border-left: 1px solid #eef2f7;
  }
</style>
"""

        # Custom investigation filtering logic (replaces PyVis default filter UI).
        # We filter edges first, then keep nodes that are endpoints of visible edges.
        js_html = f"""
<script>
  (function() {{
    if (typeof network === 'undefined' || typeof nodes === 'undefined' || typeof edges === 'undefined') {{
      console.warn('PyVis network variables not found.');
      return;
    }}

    // Keep immutable copies of the full graph state.
    const allNodes = nodes.get();
    const allEdges = edges.get();

    // Pre-computed sets from Python for faster filtering.
    const FRAUD_CONNECTED = new Set({list(sorted(fraud_connected_nodes))});
    const SHARED_VEHICLES = new Set({list(sorted(shared_vehicle_nodes))});

    // Helper: compute a stable undirected edge key.
    function edgeKey(a, b) {{
      return a < b ? a + '||' + b : b + '||' + a;
    }}

    // Helper: re-render graph with filtered nodes/edges.
    function applyFilteredGraph(filteredNodes, filteredEdges) {{
      nodes.clear();
      edges.clear();
      nodes.add(filteredNodes);
      edges.add(filteredEdges);
      try {{ network.fit({{ animation: false }}); }} catch (e) {{}}
    }}

    function getChecked(id) {{
      const el = document.getElementById(id);
      return el ? el.checked : false;
    }}

    function getNumber(id, fallback) {{
      const el = document.getElementById(id);
      if (!el) return fallback;
      const v = parseFloat(el.value);
      return Number.isFinite(v) ? v : fallback;
    }}

    function updateSliderLabel() {{
      const slider = document.getElementById('f_nodeLimit');
      const label = document.getElementById('f_nodeLimitValue');
      if (slider && label) label.textContent = slider.value;
    }}

    function resetFilters() {{
      const ids = [
        'f_showFraud','f_sharedVehicles','f_suspiciousNodes',
        'f_frequentLocations','f_highClaimEdges'
      ];
      ids.forEach(x => {{ const el = document.getElementById(x); if (el) el.checked = false; }});
      const degree = document.getElementById('f_degreeThreshold'); if (degree) degree.value = '0.05';
      const locN = document.getElementById('f_locationN'); if (locN) locN.value = '5';
      const claimT = document.getElementById('f_claimThreshold'); if (claimT) claimT.value = '50000';
      const slider = document.getElementById('f_nodeLimit'); if (slider) slider.value = Math.min(500, allNodes.length);
      updateSliderLabel();
      applyFilteredGraph(allNodes, allEdges);
    }}

    function applyFilters() {{
      updateSliderLabel();

      const showFraud = getChecked('f_showFraud');
      const sharedVehicles = getChecked('f_sharedVehicles');
      const suspiciousOnly = getChecked('f_suspiciousNodes');
      const frequentLocs = getChecked('f_frequentLocations');
      const highClaimEdges = getChecked('f_highClaimEdges');

      const degreeThreshold = getNumber('f_degreeThreshold', 0.05);
      const locationN = getNumber('f_locationN', 5);
      const claimThreshold = getNumber('f_claimThreshold', 50000);
      const nodeLimit = parseInt(getNumber('f_nodeLimit', 500), 10);

      // 1) Filter edges based on fraud/high-claim toggles.
      let filteredEdges = allEdges.filter(e => {{
        if (showFraud && !(e.fraud_flag === 1)) return false;
        if (highClaimEdges && !(e.claim_amount > claimThreshold)) return false;
        return true;
      }});

      // 2) Determine node set from visible edges.
      const nodeSet = new Set();
      filteredEdges.forEach(e => {{ nodeSet.add(e.from); nodeSet.add(e.to); }});

      // 3) Apply node-based filters (fraud-connected / shared vehicles / suspicious / frequent locations).
      let filteredNodes = allNodes.filter(n => nodeSet.has(n.id));

      if (showFraud) {{
        filteredNodes = filteredNodes.filter(n => FRAUD_CONNECTED.has(n.id));
      }}

      if (sharedVehicles) {{
        // Keep shared vehicle nodes plus their neighbors in the currently visible edge set.
        const keep = new Set();
        filteredEdges.forEach(e => {{
          if (SHARED_VEHICLES.has(e.from) || SHARED_VEHICLES.has(e.to)) {{
            keep.add(e.from); keep.add(e.to);
          }}
        }});
        filteredNodes = filteredNodes.filter(n => keep.has(n.id));
        filteredEdges = filteredEdges.filter(e => keep.has(e.from) && keep.has(e.to));
      }}

      if (suspiciousOnly) {{
        filteredNodes = filteredNodes.filter(n => (n.degree_centrality || 0) > degreeThreshold);
        const keep = new Set(filteredNodes.map(n => n.id));
        filteredEdges = filteredEdges.filter(e => keep.has(e.from) && keep.has(e.to));
      }}

      if (frequentLocs) {{
        filteredNodes = filteredNodes.filter(n => {{
          if (n.entity_type !== 'location') return true;
          return (n.incident_count || 0) > locationN;
        }});
        const keep = new Set(filteredNodes.map(n => n.id));
        filteredEdges = filteredEdges.filter(e => keep.has(e.from) && keep.has(e.to));
      }}

      // 4) Node limit: keep top-N nodes by degree_centrality (investigation focus control).
      filteredNodes.sort((a, b) => (b.degree_centrality || 0) - (a.degree_centrality || 0));
      filteredNodes = filteredNodes.slice(0, Math.max(10, nodeLimit));

      const keepFinal = new Set(filteredNodes.map(n => n.id));
      filteredEdges = filteredEdges.filter(e => keepFinal.has(e.from) && keepFinal.has(e.to));

      applyFilteredGraph(filteredNodes, filteredEdges);
    }}

    // Wire up UI events.
    const applyBtn = document.getElementById('btn_applyFilters');
    const resetBtn = document.getElementById('btn_resetFilters');
    const slider = document.getElementById('f_nodeLimit');
    if (applyBtn) applyBtn.addEventListener('click', applyFilters);
    if (resetBtn) resetBtn.addEventListener('click', resetFilters);
    if (slider) slider.addEventListener('input', updateSliderLabel);

    // Initialize slider label and render full graph.
    updateSliderLabel();
    applyFilteredGraph(allNodes, allEdges);
  }})();
</script>
"""

        # Inject the structured investigator UI right after the <body> tag.
        body_index = html_content.find("<body")
        if body_index != -1:
            body_end = html_content.find(">", body_index) + 1
            html_content = (
                html_content[:body_end]
                + css_html
                + title_html
                + panel_html
                + legend_html
                + js_html
                + html_content[body_end:]
            )

        # Save HTML file with required name
        output_file = "fraud_investigation_network.html"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html_content)

        print(f"✅ Interactive investigation network saved as: {output_file}")
        print()

        # Auto-open in browser
        try:
            file_path = os.path.abspath(output_file)
            webbrowser.open(f'file://{file_path}')
            print(f"🌐 Investigation dashboard automatically opened in your browser")
        except Exception as e:
            print(f"⚠️  Could not auto-open browser. Please open {output_file} manually.")

        print()
        print("🎯 Visualization Features Enabled:")
        print("  • Physics-based layout")
        print("  • Hover tooltips")
        print("  • Node filtering menu")
        print("  • Zoom and pan")
        print()

        return output_file

    def _create_investigation_tooltip(self, node: str, node_attrs: Dict, subgraph: nx.Graph) -> str:
        """
        Create a detailed investigation tooltip for a node.

        Args:
            node (str): Node identifier
            node_attrs (Dict): Node attributes
            subgraph (nx.Graph): Investigation subgraph

        Returns:
            str: Formatted tooltip HTML
        """
        entity_type = node_attrs.get('entity_type', 'unknown')
        degree_centrality = self.metrics['degree'].get(node, 0)
        shared_score = node_attrs.get('shared_entity_score', 0)
        connections = subgraph.degree(node)

        # Determine risk level
        if degree_centrality > 0.002:
            risk_level = "HIGH"
            risk_emoji = "🚨"
        elif degree_centrality > 0.001:
            risk_level = "MEDIUM"
            risk_emoji = "⚠️"
        else:
            risk_level = "LOW"
            risk_emoji = "✅"

        # Entity-specific information
        if entity_type == 'policy_holder':
            policy_id = node_attrs.get('policy_id', 'Unknown')
            return f"""👤 POLICY HOLDER INVESTIGATION
═══════════════════════════════
Policy ID: {policy_id}
Connected Entities: {connections}
Network Influence: {degree_centrality:.4f}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• Check for unusual claim patterns
• Verify vehicle ownership legitimacy
• Cross-reference with incident locations
• High connectivity may indicate fraud ring involvement"""

        elif entity_type == 'vehicle':
            make = node_attrs.get('vehicle_make', 'Unknown')
            model = node_attrs.get('vehicle_model', 'Unknown')
            year = node_attrs.get('vehicle_year', 'Unknown')
            return f"""🚗 VEHICLE INVESTIGATION
═══════════════════════════════
Vehicle: {make} {model} ({year})
Number of Claims: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• Used by {connections} different policy holders
• High shared usage may indicate fraud ring
• Verify vehicle identification numbers
• Check for staged accidents"""

        elif entity_type == 'location':
            city = node_attrs.get('incident_city', 'Unknown')
            state = node_attrs.get('incident_state', 'Unknown')
            return f"""📍 LOCATION INVESTIGATION
═══════════════════════════════
Location: {city}, {state}
Number of Incidents: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• Multiple accidents at same location
• May indicate staged accident location
• Verify location legitimacy
• Check for repair shop connections"""

        elif entity_type == 'zip_code':
            zip_code = node_attrs.get('insured_zip', 'Unknown')
            return f"""📮 GEOGRAPHIC AREA INVESTIGATION
═══════════════════════════════
ZIP Code: {zip_code}
Policy Holders: {connections}
Shared Entity Score: {shared_score}
Fraud Risk Level: {risk_level} {risk_emoji}

Investigation Notes:
• High concentration of policy holders
• May indicate geographic fraud ring
• Cross-reference with incident locations
• Check for organized fraud patterns"""

        else:
            return f"Unknown Entity: {node}"

    def _generate_investigation_insights(self, subgraph: nx.Graph) -> Dict[str, int]:
        """
        Generate investigation insights for the dashboard.

        Args:
            subgraph (nx.Graph): Investigation subgraph

        Returns:
            Dict[str, int]: Dictionary of investigation metrics
        """
        suspicious_set = set(self.suspicious_nodes)

        total_policies = len([
            n for n in subgraph.nodes()
            if self.graph.nodes[n].get('entity_type') == 'policy_holder'
        ])

        suspicious_policies = len([
            n for n in subgraph.nodes()
            if n in suspicious_set and self.graph.nodes[n].get('entity_type') == 'policy_holder'
        ])

        repeated_vehicles = len([
            n for n in subgraph.nodes()
            if self.graph.nodes[n].get('entity_type') == 'vehicle' and subgraph.degree(n) > 2
        ])

        repeated_locations = len([
            n for n in subgraph.nodes()
            if self.graph.nodes[n].get('entity_type') == 'location' and subgraph.degree(n) > 2
        ])

        return {
            'total_policies': total_policies,
            'suspicious_policies': suspicious_policies,
            'repeated_vehicles': repeated_vehicles,
            'repeated_locations': repeated_locations
        }

    def _create_dashboard_html(self, insights: Dict[str, int]) -> str:
        """
        Create the investigation dashboard HTML.

        Args:
            insights (Dict[str, int]): Investigation insights

        Returns:
            str: Dashboard HTML content
        """
        return f"""
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 15px; margin: 10px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.3);">
    <h1 style="text-align: center; margin: 0 0 20px 0; font-size: 2.5em; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
        🔍 FRAUD INVESTIGATION DASHBOARD
    </h1>

    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
        <!-- Investigation Insights Panel -->
        <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
            <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">📊 INVESTIGATION INSIGHTS</h3>
            <div style="font-size: 0.95em; line-height: 1.6;">
                <div style="margin-bottom: 8px;"><strong>Total Policies Analyzed:</strong> {insights['total_policies']}</div>
                <div style="margin-bottom: 8px;"><strong>Suspicious Policies:</strong> <span style="color: #ff6b6b; font-weight: bold;">{insights['suspicious_policies']}</span></div>
                <div style="margin-bottom: 8px;"><strong>Repeated Vehicles Detected:</strong> <span style="color: #ffd93d; font-weight: bold;">{insights['repeated_vehicles']}</span></div>
                <div style="margin-bottom: 8px;"><strong>Repeated Locations Detected:</strong> <span style="color: #6bcf7f; font-weight: bold;">{insights['repeated_locations']}</span></div>
            </div>
        </div>

        <!-- Graph Legend Panel -->
        <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
            <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">📖 GRAPH LEGEND</h3>
            <div style="font-size: 0.9em; line-height: 1.5;">
                <div style="margin-bottom: 6px;"><span style="color: #3498db; font-weight: bold;">👤 Blue dots</span> → Policy Holders</div>
                <div style="margin-bottom: 6px;"><span style="color: #27ae60; font-weight: bold;">🚗 Green triangles</span> → Vehicles</div>
                <div style="margin-bottom: 6px;"><span style="color: #e74c3c; font-weight: bold;">📍 Red squares</span> → Accident Locations</div>
                <div style="margin-bottom: 6px;"><span style="color: #f39c12; font-weight: bold;">📮 Orange diamonds</span> → ZIP Codes</div>
                <div style="margin-bottom: 6px;"><span style="color: #e74c3c; font-weight: bold;">🔴 Red edges</span> → Fraud flagged claims</div>
                <div style="margin-bottom: 6px;"><span style="color: #95a5a6; font-weight: bold;">⚪ Gray edges</span> → Normal claims</div>
                <div style="margin-top: 8px; padding: 6px; background: rgba(255,255,255,0.2); border-radius: 5px;">
                    <strong>Large nodes</strong> = Frequently appearing<br>
                    <strong>Red borders</strong> = High fraud risk
                </div>
            </div>
        </div>
    </div>

    <!-- Investigation Explanation Panel -->
    <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px);">
        <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">🧠 FRAUD PATTERN EXPLANATION</h3>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; font-size: 0.9em; line-height: 1.5;">
            <div>
                <h4 style="color: #6bcf7f; margin: 0 0 8px 0;">Understanding Fraud Patterns:</h4>
                <ul style="margin: 0; padding-left: 20px;">
                    <li><strong>Shared vehicle:</strong> Multiple policies connected to the same vehicle</li>
                    <li><strong>Repeated location:</strong> Many accidents at the same location</li>
                    <li><strong>High connectivity:</strong> Entities linked to many claims</li>
                    <li><strong>Fraud ring:</strong> Tightly connected cluster of claims</li>
                </ul>
            </div>
            <div>
                <h4 style="color: #6bcf7f; margin: 0 0 8px 0;">Investigation Actions:</h4>
                <ul style="margin: 0; padding-left: 20px;">
                    <li>🖱️ <strong>Hover</strong> for detailed investigation info</li>
                    <li>🔍 <strong>Zoom</strong> to examine specific areas</li>
                    <li>✋ <strong>Drag</strong> nodes to rearrange layout</li>
                    <li>🎯 <strong>Click</strong> to highlight connections</li>
                </ul>
            </div>
        </div>
    </div>

    <!-- Investigation Controls Panel -->
    <div style="background: rgba(255,255,255,0.1); padding: 15px; border-radius: 10px; backdrop-filter: blur(10px); margin-top: 15px;">
        <h3 style="color: #ffd700; margin-top: 0; font-size: 1.3em;">🛠 INVESTIGATION CONTROLS</h3>
        <div style="text-align: center;">
            <button onclick="network.setOptions({{physics:false}})" style="padding:10px 20px; margin:5px; border:none; border-radius:6px; background:#ffcc00; font-weight:bold; cursor:pointer;">
                ❄️ Freeze Layout
            </button>
            <button onclick="network.setOptions({{physics:true}})" style="padding:10px 20px; margin:5px; border:none; border-radius:6px; background:#28a745; color:white; font-weight:bold; cursor:pointer;">
                ⚡ Enable Physics
            </button>
            <button onclick="alert('🎯 INVESTIGATION GUIDE:\\n\\n1. Look for red-bordered nodes (high risk)\\n2. Follow red edges (fraud claims)\\n3. Find shared vehicles (green triangles)\\n4. Check repeated locations (red squares)\\n5. Identify clusters of connected entities\\n\\n💡 Use zoom and drag to explore patterns!')" style="padding:10px 20px; margin:5px; border:none; border-radius:6px; background:#007bff; color:white; font-weight:bold; cursor:pointer;">
                📋 Start Guided Tour
            </button>
        </div>
    </div>
</div>
"""

    def print_analysis_summary(self) -> Dict[str, Any]:
        """
        Print a comprehensive summary of the fraud analysis results.

        Returns:
            Dict[str, Any]: Summary statistics
        """
        print("=" * 80)
        print("STEP 10: ANALYSIS SUMMARY")
        print("=" * 80)

        # Basic network statistics
        total_nodes = self.graph.number_of_nodes()
        total_edges = self.graph.number_of_edges()

        # Count confirmed fraud policies (policy nodes colored red)
        fraud_policies = [
            n for n, attrs in self.graph.nodes(data=True)
            if attrs.get("entity_type") == "policy_holder" and attrs.get("fraud_policy", False)
        ]

        # Rank suspicious nodes by degree centrality and show the top 5
        degree = self.metrics.get("degree", {}) if self.metrics else {}
        suspicious_nodes = self.suspicious_nodes or []
        top_suspicious = sorted(
            suspicious_nodes,
            key=lambda n: degree.get(n, 0.0),
            reverse=True
        )[:5]

        print("📊 INVESTIGATION SUMMARY:")
        print("-" * 40)
        print(f"Number of nodes: {total_nodes}")
        print(f"Number of edges: {total_edges}")
        print(f"Number of fraud policies: {len(fraud_policies)}")
        print()

        print("🟡 Top 5 suspicious nodes by degree centrality (degree_centrality > 0.05):")
        print("-" * 40)
        if not top_suspicious:
            print("No nodes exceeded the suspicious centrality threshold.")
        else:
            for i, node in enumerate(top_suspicious, 1):
                attrs = self.graph.nodes[node]
                entity_type = attrs.get("entity_type", "unknown")
                label = attrs.get("label", str(node))
                if entity_type == "policy_holder":
                    entity_id = attrs.get("policy_id", node)
                elif entity_type == "vehicle":
                    entity_id = attrs.get("vehicle_id", node)
                elif entity_type == "location":
                    entity_id = attrs.get("location_id", node)
                elif entity_type == "zip_code":
                    entity_id = attrs.get("zip_id", node)
                else:
                    entity_id = node
                print(f"{i}. {entity_id} | type={entity_type} | degree_centrality={degree.get(node, 0.0):.4f} | label={label}")
        print()

        return {
            "total_nodes": total_nodes,
            "total_edges": total_edges,
            "fraud_policies": len(fraud_policies),
            "suspicious_nodes": len(suspicious_nodes),
            "top_suspicious_nodes": top_suspicious,
        }

    def run_complete_analysis(self) -> Dict[str, Any]:
        """
        Execute the complete fraud investigation analysis pipeline.

        Returns:
            Dict[str, Any]: Complete analysis results
        """
        print("🚀 FRAUD INVESTIGATION SYSTEM - COMPLETE ANALYSIS")
        print("=" * 80)
        print()

        try:
            # Step 1: Load dataset
            dataset = self.load_dataset()

            # Step 2: Initialize graph
            graph = self.initialize_graph()

            # Step 3: Create nodes
            entity_nodes = self.create_nodes(graph, dataset)

            # Step 4: Create edges
            self.create_edges(graph, dataset)

            # Step 5: Compute metrics
            metrics = self.compute_graph_metrics(graph)

            # Step 6: Detect suspicious entities
            suspicious_nodes = self.detect_suspicious_entities(graph, metrics)

            # Step 7: Create interactive visualization (full graph)
            network_file = self.create_interactive_investigation_graph(graph)

            # Step 8: Print investigation summary
            summary = self.print_analysis_summary()

            print("=" * 80)
            print("✅ FRAUD INVESTIGATION ANALYSIS COMPLETED")
            print("=" * 80)
            print()
            print("🎯 Key Findings:")
            print(f"  • Number of nodes: {summary['total_nodes']}")
            print(f"  • Number of edges: {summary['total_edges']}")
            print(f"  • Number of fraud policies: {summary['fraud_policies']}")
            print(f"  • Suspicious nodes (degree_centrality > 0.05): {summary['suspicious_nodes']}")
            print()
            print("📊 Interactive network saved as: fraud_investigation_network.html")
            print("🌐 Open this file in a web browser to explore the fraud investigation network")
            print()

            return {
                'summary': summary,
                'network_file': network_file,
                'suspicious_nodes': suspicious_nodes
            }

        except FileNotFoundError as e:
            print(f"❌ Error: Dataset file not found - {e}")
            print("Please ensure the graph dataset exists at:")
            print("data/processed/insurance_claims_graph_dataset.csv")
            raise
        except Exception as e:
            # Print a detailed error for easier debugging in different environments.
            print(f"❌ Error during analysis: {repr(e)}")
            print("Please check the data and try again.")
            raise


def main():
    """
    Main function to execute the fraud investigation system.
    """
    # Initialize the fraud investigation system
    investigation_system = FraudInvestigationSystem()

    # Run the complete analysis
    try:
        results = investigation_system.run_complete_analysis()
        print("\n🎉 Fraud investigation completed successfully!")
        return results
    except Exception as e:
        print(f"\n❌ Fraud investigation failed: {e}")
        return None


if __name__ == "__main__":
    main()
